In [ ]:
import pandas as pd
from google.colab import drive
import email
from email.utils import parsedate_to_datetime
import re
import json
import time

#Obtención del Dataset de mails

Conexión Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


El dataset de **Enron** es una base de datos pública que contiene más de 500.000 correos electrónicos y datos financieros de unos 150 empleados de la antigua corporación energética Enron. Fue incautado por la Comisión Federal Reguladora de Energía de EE. UU. (FERC) y liberado tras la quiebra de la empresa en 2001.

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/TFM/emails.csv", engine="python", on_bad_lines="skip")

In [ ]:
len(df)

843222

Demasiados correos, hay ruido, correos mal etiquetados, duplicados.. Filtramos solo los que tienen un patrón válido

In [ ]:
patron_valido = df["file"].str.match(r'^[a-z\-]+/.+/\d+\.$', na=False)
print(patron_valido.sum(), "filas válidas de", len(df))

513571 filas válidas de 843222


In [ ]:
df = df[patron_valido].reset_index(drop=True)
print(len(df))

513571


In [ ]:
df.head(5)

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [ ]:
df.sample(10)

,file,message
21669,beck-s/all_documents/104.,Message-ID: <16899318.1075849813334.JavaMail.e...
110073,farmer-d/personal/265.,Message-ID: <31191556.1075854147327.JavaMail.e...
6750,arnold-j/sent_items/457.,Message-ID: <10009134.1075852723360.JavaMail.e...
446661,smith-m/all_documents/249.,Message-ID: <8379935.1075863626195.JavaMail.ev...
132843,germany-c/sent_items/861.,Message-ID: <17413323.1075840529000.JavaMail.e...
260243,kitchen-l/deleted_items/149.,Message-ID: <6183167.1075855286792.JavaMail.ev...
93533,dickson-s/all_documents/17.,Message-ID: <740677.1075853650308.JavaMail.eva...
60853,dasovich-j/all_documents/27852.,Message-ID: <25875982.1075849266833.JavaMail.e...
379209,rogers-b/discussion_threads/562.,Message-ID: <4199391.1075857192320.JavaMail.ev...
229430,kean-s/all_documents/5101.,Message-ID: <13866057.1075847591848.JavaMail.e...


Dos columnas: **file**, con la ruta de origen del correo (empleado/carpeta/número), y **message**, con el correo completo en bruto (metadatos + cabeceras + cuerpo del mensaje)

In [ ]:
df.to_csv("/content/drive/MyDrive/TFM/Enron_emails.csv", index=False)

In [ ]:
empleados = df["file"].str.split("/").str[0].unique()
print(len(empleados), "empleados")

149 empleados


Hay 149 empleados, vamos a filtrar los que sen más prometedores para estudiar.

In [ ]:
conteo = df["file"].str.split("/").str[0].value_counts()
print(conteo.head(10))

file
kaminski-v      28465
dasovich-j      28097
kean-s          25249
mann-k          23381
jones-t         19950
shackleton-s    18687
taylor-m        13867
farmer-d        13032
germany-c       12436
beck-s          11827
Name: count, dtype: int64


Al ver la cantidad de difernetes carpetas de los empleados quiere decir que cada uno tiene distribuciones diferente.

In [ ]:
carpetas = df["file"].str.split("/").str[1].unique()
print(len(carpetas))
print(carpetas)

1410
['_sent_mail' 'all_documents' 'contacts' ... 'tq' 'tradecounts' 'tss']


Vamos a filtrar las carpteas, así aseguramos contenido escrito por esa persona, no correos recibidos de otros que pueden ser spam o publicidad.

In [ ]:
carpetas_por_empleado = df.groupby(df["file"].str.split("/").str[0])["file"].apply(
    lambda x: set(x.str.split("/").str[1])
)

todas_las_carpetas = [c for carpetas in carpetas_por_empleado for c in carpetas]
frecuencia_carpetas = pd.Series(todas_las_carpetas).value_counts()

print(frecuencia_carpetas.head(15))

inbox                 136
sent_items            135
deleted_items         135
all_documents         110
discussion_threads     93
sent                   89
notes_inbox            82
_sent_mail             78
calendar               70
contacts               45
personal               38
tasks                  35
to_do                  34
eol                    11
prc                    10
Name: count, dtype: int64


In [ ]:
solo_enviados = df[df["file"].str.contains("_sent_mail")]
conteo_enviados = solo_enviados["file"].str.split("/").str[0].value_counts()
print(conteo_enviados.head(10))

file
mann-k        4220
kaminski-v    3464
germany-c     1659
bass-e        1409
symes-k       1323
fossum-d      1099
beck-s        1093
scott-s        991
love-p         886
delainey-d     875
Name: count, dtype: int64


##Empleados más prometedors

In [ ]:
empleados_top = conteo.head(10).index.tolist()

for empleado in empleados_top:
    carpetas = df[df["file"].str.startswith(empleado + "/")]["file"].str.split("/").str[1].value_counts()
    print(carpetas.get("_sent_mail", 0))
    print(carpetas.get("sent", 0))
    print(carpetas.get("sent_items", 0))
    enviados = (carpetas.get("_sent_mail", 0)+carpetas.get("sent", 0)+carpetas.get("sent_items", 0))
    print(carpetas.get('notes_inbox',0))
    print(carpetas.get('inbox',0))
    recibidos = carpetas.get("inbox", 0)+carpetas.get("notes_inbox", 0)
    print(f"{empleado}: enviados={enviados}, recibidos={recibidos}, total={enviados+recibidos}")

3464
3463
1696
223
560
kaminski-v: enviados=8623, recibidos=783, total=9406
0
3925
1434
7138
1393
dasovich-j: enviados=5359, recibidos=8531, total=13890
0
1315
458
265
91
kean-s: enviados=1773, recibidos=356, total=2129
4220
4440
266
695
345
mann-k: enviados=8926, recibidos=1040, total=9966
0
3810
313
5095
957
jones-t: enviados=4123, recibidos=6052, total=10175
0
3774
633
3335
1058
shackleton-s: enviados=4407, recibidos=4393, total=8800
0
1857
547
2755
819
taylor-m: enviados=2404, recibidos=3574, total=5978
747
710
258
7
85
farmer-d: enviados=1715, recibidos=92, total=1807
1659
2116
1353
92
326
germany-c: enviados=5128, recibidos=418, total=5546
1093
1099
482
190
749
beck-s: enviados=2674, recibidos=939, total=3613


##Comprobación Duplicado

Extrae los campos estructurados (from_addr, subject, date, body) de un correo en bruto (message). Es la función base de la que dependen casi todas las demás. Necesaria.

Normaliza el asunto quitando prefijos "Re:"/"Fwd:" repetidos. Necesaria — es la base para poder agrupar mensajes del mismo hilo.

In [ ]:
import email
from email.utils import parsedate_to_datetime
import re

def descomponer_correo(row):
    msg = email.message_from_string(row["message"])
    return pd.Series({
        "from_addr": msg.get("From"),
        "subject": msg.get("Subject"),
        "date": msg.get("Date"),
        "body": msg.get_payload(),
    })

Función de exploración/diagnóstico, no de producción: compara dos carpetas de un empleado para detectar si contienen correos duplicados (comparando fecha+asunto+inicio de cuerpo). Se usó para descubrir que _sent_mail duplicaba a sent. Necesaria para justificar la decisión en la memoria, pero no forma parte del pipeline final que construye el dataset — es una herramienta de investigación puntual.

In [ ]:
def comprobar_duplicado_generico(empleado, carpeta_a, carpeta_b):
    df_check = df[df["file"].str.startswith(empleado + "/")].copy()
    df_check["carpeta"] = df_check["file"].str.split("/").str[1]
    df_check = df_check[df_check["carpeta"].isin([carpeta_a, carpeta_b])]

    parsed = df_check.apply(descomponer_correo, axis=1)
    df_check = pd.concat([df_check, parsed], axis=1)

    grp_a = df_check[df_check["carpeta"] == carpeta_a]
    grp_b = df_check[df_check["carpeta"] == carpeta_b]

    clave_a = set(zip(grp_a["date"], grp_a["subject"], grp_a["body"].str[:100]))
    clave_b = set(zip(grp_b["date"], grp_b["subject"], grp_b["body"].str[:100]))

    coinciden = clave_a & clave_b

    print(f"--- {empleado}: {carpeta_a} vs {carpeta_b} ---")
    print(f"{carpeta_a}: {len(grp_a)}, {carpeta_b}: {len(grp_b)}, coinciden: {len(coinciden)}")
    print()

In [ ]:
comprobar_duplicado_generico("mann-k", "sent", "_sent_mail")
comprobar_duplicado_generico("beck-s", "sent", "_sent_mail")
comprobar_duplicado_generico("mann-k", "sent", "sent_items")
comprobar_duplicado_generico("beck-s", "sent", "sent_items")
comprobar_duplicado_generico("mann-k", "inbox", "notes_inbox")
comprobar_duplicado_generico("beck-s", "inbox", "notes_inbox")

--- mann-k: sent vs _sent_mail ---
sent: 4440, _sent_mail: 4220, coinciden: 4210

--- beck-s: sent vs _sent_mail ---
sent: 1099, _sent_mail: 1093, coinciden: 1093

--- mann-k: sent vs sent_items ---
sent: 4440, sent_items: 266, coinciden: 0

--- beck-s: sent vs sent_items ---
sent: 1099, sent_items: 482, coinciden: 0

--- mann-k: inbox vs notes_inbox ---
inbox: 345, notes_inbox: 695, coinciden: 0

--- beck-s: inbox vs notes_inbox ---
inbox: 749, notes_inbox: 190, coinciden: 0



In [ ]:
empleados_top = conteo.head(10).index.tolist()

for empleado in empleados_top:
    carpetas = df[df["file"].str.startswith(empleado + "/")]["file"].str.split("/").str[1].value_counts()
    print(carpetas.get("sent", 0))
    print(carpetas.get("sent_items", 0))
    enviados = (carpetas.get("sent", 0)+carpetas.get("sent_items", 0))
    print(carpetas.get('notes_inbox',0))
    print(carpetas.get('inbox',0))
    recibidos = carpetas.get("inbox", 0)+carpetas.get("notes_inbox", 0)
    print(f"{empleado}: enviados={enviados}, recibidos={recibidos}, total={enviados+recibidos}")

3463
1696
223
560
kaminski-v: enviados=5159, recibidos=783, total=5942
3925
1434
7138
1393
dasovich-j: enviados=5359, recibidos=8531, total=13890
1315
458
265
91
kean-s: enviados=1773, recibidos=356, total=2129
4440
266
695
345
mann-k: enviados=4706, recibidos=1040, total=5746
3810
313
5095
957
jones-t: enviados=4123, recibidos=6052, total=10175
3774
633
3335
1058
shackleton-s: enviados=4407, recibidos=4393, total=8800
1857
547
2755
819
taylor-m: enviados=2404, recibidos=3574, total=5978
710
258
7
85
farmer-d: enviados=968, recibidos=92, total=1060
2116
1353
92
326
germany-c: enviados=3469, recibidos=418, total=3887
1099
482
190
749
beck-s: enviados=1581, recibidos=939, total=2520


Nos decidimos por...

##Construcción del dataset de beck-s

In [ ]:
carpetas_validas = ["sent", "sent_items", "inbox", "notes_inbox"]

df_beck = df[
    df["file"].str.split("/").str[0].eq("beck-s") &
    df["file"].str.split("/").str[1].isin(carpetas_validas)
].reset_index(drop=True)

df_beck_descompuesto = df_beck.apply(descomponer_correo, axis=1)
df_beck = pd.concat([df_beck, df_beck_descompuesto], axis=1)
df_beck["date_convert"] = pd.to_datetime(df_beck["date"].apply(parsedate_to_datetime), utc=True)
df_beck = df_beck.sort_values("date_convert").reset_index(drop=True)

print(len(df_beck))
print(df_beck["file"].str.split("/").str[1].value_counts())

2520
file
sent           1099
inbox           749
sent_items      482
notes_inbox     190
Name: count, dtype: int64


In [ ]:
df_beck.head(5)

,file,message,from_addr,subject,date,body,date_convert
0,beck-s/sent/679.,Message-ID: <28954278.1075855912605.JavaMail.e...,sally.beck@enron.com,Re: Rotations,"Mon, 13 Dec 1999 12:45:00 -0800 (PST)",I would love to visit with you about a rotatio...,1999-12-13 20:45:00+00:00
1,beck-s/sent/678.,Message-ID: <30597407.1075855912583.JavaMail.e...,sally.beck@enron.com,Key Contacts in Your Area - December 20 - 31,"Mon, 13 Dec 1999 14:25:00 -0800 (PST)",Please give Gretel a final list of dates that ...,1999-12-13 22:25:00+00:00
2,beck-s/sent/677.,Message-ID: <27451090.1075855912560.JavaMail.e...,sally.beck@enron.com,Re: Key Contacts in Your Area - December 20 - 31,"Wed, 15 Dec 1999 06:14:00 -0800 (PST)",Thanks for the contact names. You are right -...,1999-12-15 14:14:00+00:00
3,beck-s/sent/676.,Message-ID: <19239481.1075855912539.JavaMail.e...,sally.beck@enron.com,Re: Rotations,"Thu, 16 Dec 1999 00:42:00 -0800 (PST)",Please schedule a meeting for me with Jim Cole...,1999-12-16 08:42:00+00:00
4,beck-s/sent/675.,Message-ID: <24980799.1075855912488.JavaMail.e...,sally.beck@enron.com,Re: CFA Exam and Seminar,"Thu, 16 Dec 1999 02:39:00 -0800 (PST)",Yes. I would handle this through expense reim...,1999-12-16 10:39:00+00:00


In [ ]:
df_beck.to_csv("/content/drive/MyDrive/TFM/df_beck_inicial.csv", index=False)

Observamos el problema de que en...

crear las columnas que servirán de "etiqueta" para agrupar después

In [ ]:
def limpiar_asunto(asunto):
    if not isinstance(asunto, str):
        return ""
    asunto = asunto.strip()
    while re.match(r"(?i)^(re|fwd|fw)\s*:\s*", asunto):
        asunto = re.sub(r"(?i)^(re|fwd|fw)\s*:\s*", "", asunto)
    return asunto.strip().lower()

In [ ]:
df_beck["asunto_normalizado"] = df_beck["subject"].apply(limpiar_asunto)

df_beck["clave_agrupacion"] = df_beck.apply(
    lambda row: row["asunto_normalizado"] if row["asunto_normalizado"].strip() != "" else f"__individual__{row['file']}",
    axis=1
)

print(df_beck["clave_agrupacion"].nunique())

1813


In [ ]:
df_beck.head(3)

,file,message,from_addr,subject,date,body,date_convert,asunto_normalizado,clave_agrupacion,texto_nuevo,texto_citado,clave_agrupacion_final
0,beck-s/sent/679.,Message-ID: <28954278.1075855912605.JavaMail.e...,sally.beck@enron.com,Re: Rotations,"Mon, 13 Dec 1999 12:45:00 -0800 (PST)",I would love to visit with you about a rotatio...,1999-12-13 20:45:00+00:00,rotations,rotations,I would love to visit with you about a rotatio...,Jim Cole@ENRON_DEVELOPMENT\n12/13/99 07:15 PM\...,rotations__0
1,beck-s/sent/678.,Message-ID: <30597407.1075855912583.JavaMail.e...,sally.beck@enron.com,Key Contacts in Your Area - December 20 - 31,"Mon, 13 Dec 1999 14:25:00 -0800 (PST)",Please give Gretel a final list of dates that ...,1999-12-13 22:25:00+00:00,key contacts in your area - december 20 - 31,key contacts in your area - december 20 - 31,Please give Gretel a final list of dates that ...,,key contacts in your area - december 20 - 31__0
2,beck-s/sent/677.,Message-ID: <27451090.1075855912560.JavaMail.e...,sally.beck@enron.com,Re: Key Contacts in Your Area - December 20 - 31,"Wed, 15 Dec 1999 06:14:00 -0800 (PST)",Thanks for the contact names. You are right -...,1999-12-15 14:14:00+00:00,key contacts in your area - december 20 - 31,key contacts in your area - december 20 - 31,Thanks for the contact names. You are right -...,,key contacts in your area - december 20 - 31__1


In [ ]:
patron_cita = re.compile(
    r"(-{10,}\s*Forwarded by.*?-{5,}"
    r"|-{2,}\s*Original Message\s*-{2,}"
    r"|\n\s*To:[^\n]*\n(?:[^\n]{0,80}\n){0,30}?\s*cc:[^\n]*\n(?:[^\n]{0,80}\n){0,30}?\s*Subject:)",
    re.IGNORECASE | re.DOTALL
)

def separar_texto_nuevo(body):
    if not isinstance(body, str):
        return "", ""
    match = patron_cita.search(body)
    if match:
        fragmento_previo = body[:match.start()]
        matches_blank = list(re.finditer(r"\n\s*\n", fragmento_previo))
        corte = matches_blank[-1].start() if matches_blank else match.start()
        return body[:corte].strip(), body[corte:].strip()
    return body.strip(), ""

In [ ]:
df_beck[["texto_nuevo", "texto_citado"]] = df_beck["body"].apply(
    lambda b: pd.Series(separar_texto_nuevo(b))
)

print(df_beck[["subject", "texto_nuevo"]].head(5))

                                            subject  \
0                                     Re: Rotations   
1      Key Contacts in Your Area - December 20 - 31   
2  Re: Key Contacts in Your Area - December 20 - 31   
3                                     Re: Rotations   
4                          Re: CFA Exam and Seminar   

                                         texto_nuevo  
0  I would love to visit with you about a rotatio...  
1  Please give Gretel a final list of dates that ...  
2  Thanks for the contact names.  You are right -...  
3  Please schedule a meeting for me with Jim Cole...  
4  Yes.  I would handle this through expense reim...  


Función auxiliar simple: convierte cualquier secuencia de espacios/saltos de línea en un único espacio, quita espacios al principio/final, y pasa todo a minúsculas. Esto evita que la comparación falle por diferencias triviales de formato (dos espacios vs. uno, mayúsculas vs. minúsculas).




In [ ]:
def normalizar_espacios(texto):
    return re.sub(r"\s+", " ", texto).strip().lower()

def dividir_hilos_reales(df):
    df = df.sort_values("date_convert").copy()
    nuevas_claves = []

    for asunto, grupo in df.groupby("clave_agrupacion"):
        grupo = grupo.sort_values("date_convert")
        contador_subhilo = 0
        texto_nuevo_anterior = None

        for _, fila in grupo.iterrows():
            if texto_nuevo_anterior is not None:
                huella_anterior = normalizar_espacios(texto_nuevo_anterior)[:80]
                huella_citada = normalizar_espacios(fila["texto_citado"])
                if huella_anterior not in huella_citada:
                    contador_subhilo += 1

            nuevas_claves.append((fila.name, f"{asunto}__{contador_subhilo}"))
            texto_nuevo_anterior = fila["texto_nuevo"]

    mapa_claves = dict(nuevas_claves)
    df["clave_agrupacion_final"] = df.index.map(mapa_claves)
    return df

In [ ]:
df_beck = dividir_hilos_reales(df_beck)

print("Grupos antes:", df_beck["clave_agrupacion"].nunique())
print("Grupos después:", df_beck["clave_agrupacion_final"].nunique())

Grupos antes: 1813
Grupos después: 2229


In [ ]:
def concatenar_hilo_limpio(grupo):
    grupo_ordenado = grupo.sort_values("date_convert")
    partes = []
    for _, fila in grupo_ordenado.iterrows():
        partes.append(f"[{fila['date_convert']}] {fila['from_addr']}:\n{fila['texto_nuevo']}")
    return "\n\n---\n\n".join(partes)

def construir_dataset_final_limpio(df):
    filas_finales = []
    for clave, grupo in df.groupby("clave_agrupacion_final"):
        grupo_ordenado = grupo.sort_values("date_convert")
        body_concatenado = concatenar_hilo_limpio(grupo)
        primera_fila = grupo_ordenado.iloc[0]
        ultima_fila = grupo_ordenado.iloc[-1]

        filas_finales.append({
            "subject": ultima_fila["subject"],
            "from_addr": primera_fila["from_addr"],
            "fecha_primer_mensaje": primera_fila["date_convert"],
            "fecha_ultimo_mensaje": ultima_fila["date_convert"],
            "body": body_concatenado,
            "n_mensajes": len(grupo),
        })

    return pd.DataFrame(filas_finales)

In [ ]:
df_beck_final = construir_dataset_final_limpio(df_beck)
df_beck_final = df_beck_final.sort_values("fecha_ultimo_mensaje").reset_index(drop=True)

print(len(df_beck_final))

2229


In [ ]:
df_beck_final.head()

,subject,from_addr,fecha_primer_mensaje,fecha_ultimo_mensaje,body,n_mensajes
0,Key Contacts in Your Area - December 20 - 31,sally.beck@enron.com,1999-12-13 22:25:00+00:00,1999-12-13 22:25:00+00:00,[1999-12-13 22:25:00+00:00] sally.beck@enron.c...,1
1,Re: Key Contacts in Your Area - December 20 - 31,sally.beck@enron.com,1999-12-15 14:14:00+00:00,1999-12-15 14:14:00+00:00,[1999-12-15 14:14:00+00:00] sally.beck@enron.c...,1
2,Re: Rotations,sally.beck@enron.com,1999-12-13 20:45:00+00:00,1999-12-16 08:42:00+00:00,[1999-12-13 20:45:00+00:00] sally.beck@enron.c...,2
3,Re: CFA Exam and Seminar,sally.beck@enron.com,1999-12-16 10:39:00+00:00,1999-12-16 10:39:00+00:00,[1999-12-16 10:39:00+00:00] sally.beck@enron.c...,1
4,Re: Project Aruba Closing Dinner,sally.beck@enron.com,1999-12-16 14:12:00+00:00,1999-12-16 14:12:00+00:00,[1999-12-16 14:12:00+00:00] sally.beck@enron.c...,1


In [ ]:
df_beck_final.sample(10)

,subject,from_addr,fecha_primer_mensaje,fecha_ultimo_mensaje,body,n_mensajes
1549,FW: Enron Center South (ECS) Move Back-up Plan,sally.beck@enron.com,2001-10-09 17:01:06+00:00,2001-10-09 17:01:06+00:00,[2001-10-09 17:01:06+00:00] sally.beck@enron.c...,1
1309,RE: vanessa Schulte,beth.apollo@enron.com,2001-05-07 17:54:06+00:00,2001-05-07 21:42:45+00:00,[2001-05-07 17:54:06+00:00] beth.apollo@enron....,2
1739,FW: Master Netting Agreement Assignments...Ide...,leslie.reeves@enron.com,2001-11-02 18:44:56+00:00,2001-11-02 18:44:56+00:00,[2001-11-02 18:44:56+00:00] leslie.reeves@enro...,1
739,Regional Office Summary - December,lisa.fair@enron.com,2000-12-11 10:57:00+00:00,2000-12-11 10:57:00+00:00,[2000-12-11 10:57:00+00:00] lisa.fair@enron.co...,1
21,Various Discussion Points,sally.beck@enron.com,1999-12-17 19:03:00+00:00,2000-01-04 15:40:00+00:00,[1999-12-17 19:03:00+00:00] sally.beck@enron.c...,2
1678,RE: Amanda on All A's Honor Roll,sally.beck@enron.com,2001-10-28 22:37:26+00:00,2001-10-28 22:37:26+00:00,[2001-10-28 22:37:26+00:00] sally.beck@enron.c...,1
1445,Restaurant Recommedations - Telluride,sally.beck@enron.com,2001-06-04 18:43:44+00:00,2001-06-04 18:43:44+00:00,[2001-06-04 18:43:44+00:00] sally.beck@enron.c...,1
1299,FW: Updated '98 - '01 Headcount File,brian.heinrich@enron.com,2001-05-07 16:47:15+00:00,2001-05-07 17:56:36+00:00,[2001-05-07 16:47:15+00:00] brian.heinrich@enr...,2
1187,Net Works Floor Meetings - DATES AND LOCATIONS...,sally.beck@enron.com,2001-04-09 20:31:00+00:00,2001-04-09 20:31:00+00:00,[2001-04-09 20:31:00+00:00] sally.beck@enron.c...,1
1464,GCP Status - Sally & James,beth.apollo@enron.com,2001-06-06 15:33:35+00:00,2001-06-06 15:33:35+00:00,[2001-06-06 15:33:35+00:00] beth.apollo@enron....,1


In [ ]:
ejemplo_simple = df_beck_final.iloc[0]
ejemplo_multiple = df_beck_final.iloc[2]

print("=== EJEMPLO CON 1 MENSAJE ===")
print(f"Subject: {ejemplo_simple['subject']}")
print(f"From: {ejemplo_simple['from_addr']}")
print(f"Body:\n{ejemplo_simple['body']}")
print()
print("=== EJEMPLO CON VARIOS MENSAJES ===")
print(f"Subject: {ejemplo_multiple['subject']}")
print(f"N_mensajes: {ejemplo_multiple['n_mensajes']}")
print(f"Body:\n{ejemplo_multiple['body']}")

=== EJEMPLO CON 1 MENSAJE ===
Subject: Key Contacts in Your Area - December 20 - 31
From: sally.beck@enron.com
Body:
[1999-12-13 22:25:00+00:00] sally.beck@enron.com:
Please give Gretel a final list of dates that you plan to be out of the 
office from December 20 through the end of the year.  For the dates that you 
will be out, please provide the name and number of the key contact person in 
your group that will be in the office on the days that you are out.  

As I have in past years, I will compile a list of key contacts through the 
holiday period that we can use internally within Energy Operations.  I will 
also provide this list to others outside Energy Operations who may needa  
contact list during this time period.  

Please get this info to Gretel by Wednesday at noon, so that I have time to 
finalize the contact list before I am out on vacation.  

FYI:  I plan to be out of the office the week of December 20.  I will be at 
home during that week and can be reached there if ne

Correos con más de un mensaje

In [ ]:
hilos_multiples = df_beck_final[df_beck_final["n_mensajes"] > 1]
print(len(hilos_multiples))

for i, row in hilos_multiples.head(10).iterrows():
    print("=" * 60)
    print(f"SUBJECT: {row['subject']} | N_MENSAJES: {row['n_mensajes']}")
    print("-" * 60)
    print(row["body"])
    print()

261
SUBJECT: Re: Rotations | N_MENSAJES: 2
------------------------------------------------------------
[1999-12-13 20:45:00+00:00] sally.beck@enron.com:
I would love to visit with you about a rotation.  I plan on being on vacation 
next week, but I will be working between Christmas and New Years.  What are 
your plans?  With a PRC tomorrow (Tuesday, December 14) and the crunch of 
finishing a number of items before I go on vacation, it may be tough to get 
together this week.  So get back to me with your schedule.  Thanks.  --Sally

---

[1999-12-16 08:42:00+00:00] sally.beck@enron.com:
Please schedule a meeting for me with Jim Cole for the week after Christmas.  
Thanks.

SUBJECT: T. Hall Final 1999 Exp Rpt | N_MENSAJES: 2
------------------------------------------------------------
[1999-12-17 08:18:00+00:00] sally.beck@enron.com:
Approved.  Pls Process.  Thanks.  Sally Beck

---

[1999-12-20 14:30:00+00:00] sally.beck@enron.com:
This is what we sent to A/P. Let me know if we need t

In [ ]:
hilos_multiples_6 = df_beck_final[df_beck_final["n_mensajes"] == 6]

In [ ]:
display(hilos_multiples_6)

,subject,from_addr,fecha_primer_mensaje,fecha_ultimo_mensaje,body,n_mensajes
2023,FW: 2002 Budget for Enron Net Works,sally.beck@enron.com,2001-12-31 21:37:25+00:00,2002-01-02 23:55:26+00:00,[2001-12-31 21:37:25+00:00] sally.beck@enron.c...,6


Comprobación de sospechosos al realizar las uniones

In [ ]:
sospechosos = df_beck_final[df_beck_final["body"].str.contains(r"\bSubject:", case=False, na=False, regex=True)]
print(len(sospechosos))

22


In [ ]:
for i, row in sospechosos.head(5).iterrows():
    print("=" * 60)
    print(f"SUBJECT: {row['subject']}")
    print("-" * 60)
    print(row["body"])
    print()

SUBJECT: 
------------------------------------------------------------
[2000-03-21 10:47:00+00:00] sally.beck@enron.com:
DISCUSSION DRAFT




TO:  Enron Europe Employees=20

FROM:  Fernley Dyson, Vice President and Chief Accounting Officer, EEL
  Sally Beck, Vice President =01) Global Risk Management Operations=20

SUBJECT: Trading Operations Resource for London


Enron Europe Ltd. has played a vital role over the last few years in the=20
global expansion of  Enron=01,s  business.  Now, EEL is taking a strategic =
role=20
in setting the standards for global operational performance.  Accordingly,=
=20
EEL will work closely with Global Risk Management Operations to assess our=
=20
operational policies, procedures and internal controls.  This will be a=20
collaborative effort to establish and implement standards and measurements =
to=20
be used world-wide.  This strategic initiative is critical to Enron=01,s=20
performance, in insuring that trading operations services are delivered=20
con

In [ ]:
df_beck_final["from_addr"].nunique()

302

In [ ]:
df_beck_final["n_mensajes"].describe()

,n_mensajes
count,2229.000000
mean,1.130552
std,0.387767
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,6.000000


In [ ]:
df_beck_final["n_mensajes"].value_counts().sort_index()

,count
n_mensajes,
1,1968
2,238
3,19
4,2
5,1
6,1


In [ ]:
n_palabras = df_beck_final["body"].str.split().str.len()
print(n_palabras.describe())

count     2229.000000
mean       135.882907
std        595.234761
min          3.000000
25%         27.000000
50%         62.000000
75%        135.000000
max      17175.000000
Name: body, dtype: float64


In [ ]:
df_beck_final.to_csv("/content/drive/MyDrive/TFM/dataset_beck_final.csv", index=False)

#Indexación

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_beck_final = pd.read_csv("/content/drive/MyDrive/TFM/dataset_beck_final.csv")
print(len(df_beck_final))

2229


In [ ]:
# Instalamos ChromaDB, que será nuestra base de datos vectorial
!pip install chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

##Indexación A

In [ ]:
n_palabras = df_beck_final["body"].str.split().str.len()
n_palabras_limite = df_beck_final[n_palabras > 200]
print(len(n_palabras_limite))

309


Se instala el tokenizador real del modelo de embeddings (all-MiniLM-L6-v2) mediante la librería transformers, en lugar de aproximar por número de palabras. Para cada correo del dataset, se calcula el número exacto de tokens que generaría ese tokenizador (sin truncar, para obtener el conteo completo), y se cuenta cuántos correos superan el límite real de 256 tokens que admite el modelo — esta cifra indica con precisión cuántos documentos se verían afectados por el truncamiento silencioso al generar sus embeddings.

In [ ]:
!pip install transformers -q

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

n_tokens = df_beck_final["body"].apply(lambda x: len(tokenizer.encode(x, truncation=False)))

print(n_tokens.describe())
print()
n_tokens_limite = (n_tokens > 256).sum()
print(f"Documentos que superan 256 tokens (se truncan): {n_tokens_limite} de {len(df_beck_final)} ({n_tokens_limite/len(df_beck_final)*100:.1f}%)")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


count     2229.000000
mean       212.447734
std        870.931814
min         27.000000
25%         60.000000
50%        106.000000
75%        199.000000
max      25058.000000
Name: body, dtype: float64

Documentos que superan 256 tokens (se truncan): 359 de 2229 (16.1%)


Con este dato confirmado (16,1% de tus documentos truncados), tienes ya la cifra exacta y defendible para justificar ante tu tutor por qué es necesario abordar esto con chunking.

ChromaDB, al realizar la indexación en estos 359 correos, no genera ningún error ni aviso explícito: simplemente calcula el embedding a partir de los primeros 256 tokens del texto, descartando de forma silenciosa el resto del contenido. Esto implica que, para estos documentos, el vector resultante no representa el significado del correo completo, sino únicamente el de su fragmento inicial, con el consiguiente riesgo de que información relevante situada en la parte final del correo quede fuera del alcance de la búsqueda semántica.

In [ ]:
print(df_beck_final['body'].iloc[2])

[1999-12-13 20:45:00+00:00] sally.beck@enron.com:
I would love to visit with you about a rotation.  I plan on being on vacation 
next week, but I will be working between Christmas and New Years.  What are 
your plans?  With a PRC tomorrow (Tuesday, December 14) and the crunch of 
finishing a number of items before I go on vacation, it may be tough to get 
together this week.  So get back to me with your schedule.  Thanks.  --Sally

---

[1999-12-16 08:42:00+00:00] sally.beck@enron.com:
Please schedule a meeting for me with Jim Cole for the week after Christmas.  
Thanks.


In [ ]:
correos_largos = df_beck_final[n_tokens > 256]
print(correos_largos["n_mensajes"].value_counts().sort_index())

n_mensajes
1    234
2    109
3     12
4      2
5      1
6      1
Name: count, dtype: int64


Dato revelador: 234 de los 359 correos largos (65%) tienen un solo mensaje — es decir, la mayoría del problema de truncamiento no viene de hilos con varios mensajes concatenados, sino de correos individuales que ya son largos por sí mismos.

La idea de "aprovechar los límites de mensaje del hilo" solo ayudaría directamente al 35%, para el 65% con un solo mensaje, no hay ningún límite natural de "conversación" que aprovechar, y el semantic chunking (cortar por cambio de tema dentro del propio texto) sería la única forma real de dividirlos con sentido, ya que no hay estructura de hilo que utilizar.

Esto refuerza que el semantic chunking que propuso tu tutor es más apropiado que mi idea simplificada inicial — cubre bien ambos casos (correos de un mensaje muy largo, e hilos con varios mensajes), mientras que mi propuesta solo resolvía parcialmente el problema. ¿Seguimos con la implementación de semantic chunking entonces, probándola primero con uno de estos correos largos de un solo mensaje?

###Semantic Chunking

In [ ]:
import chromadb

cliente_completos = chromadb.PersistentClient(path="/content/drive/MyDrive/TFM/chroma_beck_A")
collection_completos = cliente_completos.get_or_create_collection(
    name="beck_mails_completos",
    metadata={"hnsw:space": "cosine"}
)

print(collection_completos.count())

3684


In [ ]:
!pip install sentence-transformers -q

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def chunking_semantico(texto, percentil_corte=90):
    frases = re.split(r'(?<=[.!?])\s+', texto.strip())
    frases = [f for f in frases if f.strip()]

    if len(frases) <= 1:
        return [texto]

    embeddings_frases = modelo_embeddings.encode(frases)

    distancias = []
    for i in range(len(embeddings_frases) - 1):
        sim = np.dot(embeddings_frases[i], embeddings_frases[i+1]) / (
            np.linalg.norm(embeddings_frases[i]) * np.linalg.norm(embeddings_frases[i+1])
        )
        distancias.append(1 - sim)

    umbral = np.percentile(distancias, percentil_corte)
    puntos_corte = [i for i, d in enumerate(distancias) if d > umbral]

    chunks = []
    inicio = 0
    for punto in puntos_corte:
        chunks.append(" ".join(frases[inicio:punto+1]))
        inicio = punto + 1
    chunks.append(" ".join(frases[inicio:]))

    return chunks

In [ ]:
idx_ejemplo = correos_largos.index[0]
texto_ejemplo = df_beck_final.loc[idx_ejemplo, "body"]

chunks_prueba = chunking_semantico(texto_ejemplo)

print(f"Número de chunks generados: {len(chunks_prueba)}")
for i, c in enumerate(chunks_prueba):
    print(f"--- Chunk {i} ({len(c.split())} palabras) ---")
    print(c)
    print()

Número de chunks generados: 3
--- Chunk 0 (236 palabras) ---
[1999-12-17 16:55:00+00:00] sally.beck@enron.com:
We are working very diligently to accommodate the need for space on the 30th 
floor. Unfortunately, we are not able to do any major construction to 
densify the floor during the next several months while Enron Property 
Services is completing the build-out of levels 29, 31 and 32. In the mean 
time, we are going to build out the window spaces on 30 to increase the 
capacity on the floor. While these locations are not prime spots, people 
will need to occupy them as our business continues to grow. Once the window 
seats have been constructed (estimated to be 2-3 weeks), we will realign 
groups on the floor and people will need to move once again. I am requesting your cooperation and patience to work with Lisa Cousino while 
she works with space planners to coordinate the location of Energy Operations 
personnel on the newly-constructed Commercial floors. Susan Harrison is 
work

In [ ]:
def chunking_semantico(texto, percentil_corte=90, min_palabras=20):
    frases = re.split(r'(?<=[.!?])\s+', texto.strip())
    frases = [f for f in frases if f.strip()]

    if len(frases) <= 1:
        return [texto]

    embeddings_frases = modelo_embeddings.encode(frases)

    distancias = []
    for i in range(len(embeddings_frases) - 1):
        sim = np.dot(embeddings_frases[i], embeddings_frases[i+1]) / (
            np.linalg.norm(embeddings_frases[i]) * np.linalg.norm(embeddings_frases[i+1])
        )
        distancias.append(1 - sim)

    umbral = np.percentile(distancias, percentil_corte)
    puntos_corte = [i for i, d in enumerate(distancias) if d > umbral]

    chunks = []
    inicio = 0
    for punto in puntos_corte:
        chunks.append(" ".join(frases[inicio:punto+1]))
        inicio = punto + 1
    chunks.append(" ".join(frases[inicio:]))

    # Fusionar chunks demasiado pequeños con el siguiente
    chunks_fusionados = []
    buffer = ""
    for c in chunks:
        buffer = (buffer + " " + c).strip() if buffer else c
        if len(buffer.split()) >= min_palabras:
            chunks_fusionados.append(buffer)
            buffer = ""
    if buffer:
        if chunks_fusionados:
            chunks_fusionados[-1] += " " + buffer
        else:
            chunks_fusionados.append(buffer)

    return chunks_fusionados

In [ ]:
idx_ejemplo = correos_largos.index[0]
texto_ejemplo = df_beck_final.loc[idx_ejemplo, "body"]

chunks_prueba = chunking_semantico(texto_ejemplo)

print(f"Número de chunks generados: {len(chunks_prueba)}")
for i, c in enumerate(chunks_prueba):
    print(f"--- Chunk {i} ({len(c.split())} palabras) ---")
    print(c)
    print()

Número de chunks generados: 2
--- Chunk 0 (236 palabras) ---
[1999-12-17 16:55:00+00:00] sally.beck@enron.com:
We are working very diligently to accommodate the need for space on the 30th 
floor. Unfortunately, we are not able to do any major construction to 
densify the floor during the next several months while Enron Property 
Services is completing the build-out of levels 29, 31 and 32. In the mean 
time, we are going to build out the window spaces on 30 to increase the 
capacity on the floor. While these locations are not prime spots, people 
will need to occupy them as our business continues to grow. Once the window 
seats have been constructed (estimated to be 2-3 weeks), we will realign 
groups on the floor and people will need to move once again. I am requesting your cooperation and patience to work with Lisa Cousino while 
she works with space planners to coordinate the location of Energy Operations 
personnel on the newly-constructed Commercial floors. Susan Harrison is 
work

In [ ]:
idx_max = n_tokens.idxmax()
texto_largo = df_beck_final.loc[idx_max, "body"]
print(f"Índice: {idx_max}, tokens: {n_tokens.loc[idx_max]}")

chunks_largo = chunking_semantico(texto_largo)
print(f"Número de chunks generados: {len(chunks_largo)}")
for i, c in enumerate(chunks_largo):
    print(f"--- Chunk {i} ({len(c.split())} palabras) ---")
    print(c[:150])
    print()

Índice: 1915, tokens: 25058
Número de chunks generados: 52
--- Chunk 0 (64 palabras) ---
[2001-11-26 15:37:31+00:00] m..schmidt@enron.com:
Premiums Stay High on Enron's Near Options, And `Doubling Up' Date Looms fo=
r Tax Losses
The Wall S

--- Chunk 1 (986 palabras) ---
Morgan's Paper Anniversary
A Year After the Merger, Rosy Plans Meet Reality
The New York Times, 11/23/01
COMPANIES & FINANCE THE AMERICAS - Enron 'awa

--- Chunk 2 (478 palabras) ---
As of 4 p.m. Wednesday, Dyne=
gy shares fell $1.94 to $39.76 each.=20
On Wednesday, Dynegy issued a statement in which Chairman and Chief Executi=
ve 

--- Chunk 3 (1138 palabras) ---
Barone said he thinks a "likely scenario"=
 is that the merger formula will be renegotiated sharply down to about 0.15=
 Dynegy share for each Enron s

--- Chunk 4 (471 palabras) ---
"I feel like I have been betrayed," Mr. Rinard said in press release=
 issued by his lawyers. "I lost my savings, my plans for the future, everyt=
hin

--- Chunk 5 (24 palabras) 

In [ ]:
def dividir_por_tokens(chunk, max_tokens=225):
    tokens = tokenizer.encode(chunk, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return [chunk]

    sub_chunks = []
    for i in range(0, len(tokens), max_tokens):
        sub_tokens = tokens[i:i + max_tokens]
        sub_chunks.append(tokenizer.decode(sub_tokens))
    return sub_chunks


def chunking_semantico(texto, percentil_corte=90, min_palabras=15, max_tokens=225):
    frases = re.split(r'(?<=[.!?])\s+', texto.strip())
    frases = [f for f in frases if f.strip()]

    if len(frases) <= 1:
        return dividir_por_tokens(texto, max_tokens)

    embeddings_frases = modelo_embeddings.encode(frases)

    distancias = []
    for i in range(len(embeddings_frases) - 1):
        sim = np.dot(embeddings_frases[i], embeddings_frases[i+1]) / (
            np.linalg.norm(embeddings_frases[i]) * np.linalg.norm(embeddings_frases[i+1])
        )
        distancias.append(1 - sim)

    umbral = np.percentile(distancias, percentil_corte)
    puntos_corte = [i for i, d in enumerate(distancias) if d > umbral]

    chunks = []
    inicio = 0
    for punto in puntos_corte:
        chunks.append(" ".join(frases[inicio:punto+1]))
        inicio = punto + 1
    chunks.append(" ".join(frases[inicio:]))

    # Fusionar chunks demasiado pequeños con el siguiente
    chunks_fusionados = []
    buffer = ""
    for c in chunks:
        buffer = (buffer + " " + c).strip() if buffer else c
        if len(buffer.split()) >= min_palabras:
            chunks_fusionados.append(buffer)
            buffer = ""
    if buffer:
        if chunks_fusionados:
            chunks_fusionados[-1] += " " + buffer
        else:
            chunks_fusionados.append(buffer)

    # Subdividir por tamaño fijo cualquier chunk que aún supere el límite
    chunks_finales = []
    for c in chunks_fusionados:
        chunks_finales.extend(dividir_por_tokens(c, max_tokens))

    return chunks_finales

In [ ]:
chunks_largo = chunking_semantico(texto_largo)
print(f"Número de chunks generados: {len(chunks_largo)}")

n_tokens_chunks = [len(tokenizer.encode(c, add_special_tokens=False)) for c in chunks_largo]
print("Máximo de tokens en un chunk:", max(n_tokens_chunks))
print("¿Alguno supera 256?", any(t > 256 for t in n_tokens_chunks))

Número de chunks generados: 139
Máximo de tokens en un chunk: 228
¿Alguno supera 256? False


In [ ]:
documents = []
ids = []
metadatas = []

for idx, fila in df_beck_final.iterrows():
    n_tok = n_tokens.loc[idx]

    if n_tok <= 256:
        documents.append(fila["body"])
        ids.append(str(idx))
        metadatas.append({
            "subject": fila["subject"],
            "from_addr": fila["from_addr"],
            "fecha_primer_mensaje": str(fila["fecha_primer_mensaje"]),
            "fecha_ultimo_mensaje": str(fila["fecha_ultimo_mensaje"]),
            "n_mensajes": int(fila["n_mensajes"]),
            "es_chunk": False,
        })
    else:
        chunks = chunking_semantico(fila["body"])
        for i, chunk in enumerate(chunks):
            documents.append(chunk)
            ids.append(f"{idx}_chunk{i}")
            metadatas.append({
                "subject": fila["subject"],
                "from_addr": fila["from_addr"],
                "fecha_primer_mensaje": str(fila["fecha_primer_mensaje"]),
                "fecha_ultimo_mensaje": str(fila["fecha_ultimo_mensaje"]),
                "n_mensajes": int(fila["n_mensajes"]),
                "es_chunk": True,
                "documento_original_id": str(idx),
                "chunk_index": i,
                "total_chunks": len(chunks),
            })

print(f"Total de documentos a indexar: {len(documents)}")

Total de documentos a indexar: 3684


In [ ]:
import time

inicio_indexacion = time.time()

collection_completos.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

tiempo_indexacion = time.time() - inicio_indexacion

print(collection_completos.count())
print(f"Tiempo de indexación: {tiempo_indexacion:.2f} segundos")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 38.7MiB/s]


3684
Tiempo de indexación: 382.65 segundos


###Comprobaciones

In [ ]:
resultado1 = collection_completos.get(ids=["0"], include=["embeddings", "documents"])
print("Documento:", resultado1["documents"][0][:100])
print()
print("Dimensión del vector:", len(resultado1["embeddings"][0]))
print()
print("Primeros 10 números del vector:")
print(resultado1["embeddings"][0][:10])

Documento: [1999-12-13 22:25:00+00:00] sally.beck@enron.com:
Please give Gretel a final list of dates that you 

Dimensión del vector: 384

Primeros 10 números del vector:
[-0.08850214  0.02933304  0.03912438  0.02750664  0.02704943 -0.01313296
  0.02598798  0.03893017 -0.04608588 -0.04521352]


In [ ]:
print(n_tokens.loc[0])

232


In [ ]:
resultado2 = collection_completos.get(ids=["1915_chunk0"], include=["embeddings", "documents", "metadatas"])
print("Documento:", resultado2["documents"][0][:150])
print()
print("Metadatos:", resultado2["metadatas"][0])
print()
print("Dimensión del vector:", len(resultado2["embeddings"][0]))

Documento: [2001-11-26 15:37:31+00:00] m..schmidt@enron.com:
Premiums Stay High on Enron's Near Options, And `Doubling Up' Date Looms fo=
r Tax Losses
The Wall S

Metadatos: {'from_addr': 'm..schmidt@enron.com', 'documento_original_id': '1915', 'fecha_ultimo_mensaje': '2001-11-26 15:37:31+00:00', 'subject': 'Enron Mentions -  11/23/01', 'es_chunk': True, 'fecha_primer_mensaje': '2001-11-26 15:37:31+00:00', 'chunk_index': 0, 'n_mensajes': 1, 'total_chunks': 139}

Dimensión del vector: 384


Consultas Semánticas

In [ ]:
print(df_beck_final['body'].iloc[162])

[2000-04-18 14:17:00+00:00] sally.beck@enron.com:
We did talk about the creation of these new positions within gas support.  I 
have also discussed this at length with Brent, and agree that the shift of 
manager positions to front to back looks across each region (eliminating the 
OA and Deal Validation manager roles) makes sense.  Upon the recommendation 
of Susan and Brent, I am comfortable with appointing Jeff Gossett as the 
manager of the West region.


In [ ]:
resultado3 = collection_completos.query(
    query_texts=["Who was appointed as manager of the West region?"],
    n_results=3
)

for doc, meta, dist in zip(resultado3["documents"][0], resultado3["metadatas"][0], resultado3["distances"][0]):
    print(f"Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:200]}...")
    print()

Distancia: 0.4117
Subject: Re: Reorganization/Promotion
Body: [2000-04-18 14:17:00+00:00] sally.beck@enron.com:
We did talk about the creation of these new positions within gas support.  I 
have also discussed this at length with Brent, and agree that the shift ...

Distancia: 0.6219
Subject: Natural Gas Origination
Body: Strategic Asset  Development
Laura Luce will move from her role in the Central Region to lead an effort focused strictly on identifying and entering into long-term strategic arrangements within the Ce...

Distancia: 0.6275
Subject: Natural Gas Origination
Body: [2001-10-25 02:22:28+00:00] no.address@enron.com:
Our natural gas business continues to benefit from effective account management and resource allocation focused on identifying and responding to the n...



In [ ]:
print(df_beck_final['body'].iloc[254])

[2000-06-12 17:41:00+00:00] sally.beck@enron.com:
I am not close enough to know Bob's exact role within the group to know the 
impact of the prospect of losing him.  I am forwarding this to Brent to get 
his assessment of the impact.  I do know that the number of deals that we do 
currently has skyrocketed with EOL, so I feel sure that the group is 
stretched right now.  Again, though, Brent would best be able to address 
this.  As you are probably aware, I personnally made the commitment to Bob 
several years ago to help in managing his workload so that he is available as 
needed at home for his wife.  For Bob's sake, I would want to make sure that 
he is in a situation where that is understood and appreciated.  

Thank you for contacting me first before going to Bob.  --Sally


In [ ]:
resultado31 = collection_completos.query(
    query_texts=["Who did Sally ask to assess the impact of losing Bob?"],
    n_results=3
)

for id_doc, doc, meta, dist in zip(resultado31["ids"][0], resultado31["documents"][0], resultado31["metadatas"][0], resultado31["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f} | es_chunk: {meta.get('es_chunk')}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

ID: 254 | Distancia: 0.5393 | es_chunk: False
Subject: Re: Personnel for EBS
Body: [2000-06-12 17:41:00+00:00] sally.beck@enron.com:
I am not close enough to know Bob's exact role within the group to know the 
impact of the prospect ...

ID: 1075 | Distancia: 0.5546 | es_chunk: False
Subject: Re: Material for Commodity Logic meeting
Body: [2001-03-28 08:15:00+00:00] patti.thompson@enron.com:
Bob,

Sally is out all day w/ UT recruiting function.  She will not be back into 
the office unt...

ID: 1052 | Distancia: 0.5701 | es_chunk: False
Subject: Re: VMS Project Next Steps
Body: [2001-03-15 11:00:00+00:00] patti.thompson@enron.com:
Murray,

Sally Beck and Leslie Reeves are both on vacation this week.  Sally will need 
to discu...



In [ ]:
correos_largos = df_beck_final[n_tokens > 256]
idx_ejemplo_chunk = correos_largos.index[8]  # elegimos uno cualquiera de la lista, no el extremo

print(f"Índice: {idx_ejemplo_chunk}, tokens: {n_tokens.loc[idx_ejemplo_chunk]}")
print(df_beck_final.loc[idx_ejemplo_chunk, "subject"])
print()
print(df_beck_final.loc[idx_ejemplo_chunk, "body"][:2000])

Índice: 121, tokens: 868
Trip to London Next Week

[2000-03-14 09:58:00+00:00] sally.beck@enron.com:
Brent Price and I will be in the London office next week, arriving Monday 
morning and returning to Houston on Friday.  The purpose of my visit is in 
regard to my Global Risk Management Operations role,   I believe that Rick 
Causey met with you regarding some operations support that Brent and I will 
provide to Fernley during a transition period prior to the arrival and during 
the assimilation of Michael Jordan, the JP Morgan staffer that Fernley has 
hired.  Brent and I will be working with Fernley next week to iron out the 
details of our support for Fernley during that transition period.  We will 
also map out an Enron trading operations introduction  for Michael Jordan, 
which may involve a short stint for him in Houston.  My goal is to provide a 
strong resource to Fernley during this transition period and to work with 
Fernley to provide whatever assistance, training, and orien

In [ ]:
resultado32 = collection_completos.query(
    query_texts=["Who is Michael Jordan and why did Sally travel to London to help with his transition?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado32["ids"][0], resultado32["documents"][0], resultado32["metadatas"][0], resultado32["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f} | es_chunk: {meta.get('es_chunk')}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

ID: 614 | Distancia: 0.5298 | es_chunk: False
Subject: Re: Reporting Line for Sydney Risk Officer
Body: [2000-11-17 08:34:00+00:00] scott.mills@enron.com:
I think that it makes perfect sense now that Sydney rolls into London.  Sally 
and I have discussed...

ID: 812 | Distancia: 0.5631 | es_chunk: False
Subject: Re: Holiday contact list
Body: [2000-12-27 14:21:00+00:00] sally.beck@enron.com:
Good memo on contacts.  Thanks for copying me.  Have a wonderful vacation.   
I know that you will b...

ID: 932 | Distancia: 0.5678 | es_chunk: False
Subject: Re: Congratulations!!
Body: [2001-01-23 11:48:00+00:00] sally.beck@enron.com:
Thanks for your note.  Sorry that all we managed last week was a wave.  My 
last trip to London was ...

ID: 856_chunk0 | Distancia: 0.5726 | es_chunk: True
Subject: Re: Visit to London
Body: [ 2001 - 01 - 11 09 : 50 : 00 + 00 : 00 ] sally. beck @ enron. com : i hope that your holidays were wonderful. i thoroughly enjoyed the week of vacati...

ID: 615 | Distancia:

In [ ]:
correos_largos = df_beck_final[n_tokens > 256]
idx_ejemplo_chunk = correos_largos.index[12]  # elegimos uno cualquiera de la lista, no el extremo

print(f"Índice: {idx_ejemplo_chunk}, tokens: {n_tokens.loc[idx_ejemplo_chunk]}")
print(df_beck_final.loc[idx_ejemplo_chunk, "subject"])
print()
print(df_beck_final.loc[idx_ejemplo_chunk, "body"][:2000])

Índice: 137, tokens: 376
Astros Tickets

[2000-03-29 19:04:00+00:00] sally.beck@enron.com:
Energy Operations has two sets of season tickets for Astros games this 
season.  One set is four tickets in the field box level, and the other set is 
two tickets in the terrace level.  I have also purchased one parking spot in 
a lot that is a couple of blocks from the stadium.  I have divided the 
tickets among you, with my husband's help.  He identified "good games" (i.e. 
teams that everyone likes to see), and those are distributed equally across 
my direct reports.  Tickets for the other games have been distributed based 
your percentage of Energy Operations employees.  For most games, you will 
receive all six tickets.  You can use these tickets for yourself and for a 
small reward for a job well done by someone in your group.  The parking 
information will accompany the group of four field box tickets.  Patti is 
including a map of downtown with the parking lot pinpointed.  Your tickets 
w

In [ ]:
resultado34 = collection_completos.query(
    query_texts=["How many Astros tickets did Sally receive for the exhibition game on April 1?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado34["ids"][0], resultado34["documents"][0], resultado34["metadatas"][0], resultado34["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f} | es_chunk: {meta.get('es_chunk')}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

ID: 137_chunk0 | Distancia: 0.3668 | es_chunk: True
Subject: Astros Tickets
Body: [2000-03-29 19:04:00+00:00] sally.beck@enron.com:
Energy Operations has two sets of season tickets for Astros games this 
season. One set is four tick...

ID: 1183_chunk0 | Distancia: 0.3694 | es_chunk: True
Subject: Astros Tickets 2001 Season
Body: [ 2001 - 04 - 09 17 : 39 : 00 + 00 : 00 ] sally. beck @ enron. com : patti will be distributing astros tickets to you for the 2001 season. we used a p...

ID: 1192_chunk0 | Distancia: 0.3695 | es_chunk: True
Subject: Astros Tickets 2001 Season
Body: [ 2001 - 04 - 10 03 : 39 : 00 + 00 : 00 ] sally. beck @ enron. com : patti will be distributing astros tickets to you for the 2001 season. we used a p...

ID: 116_chunk1 | Distancia: 0.4198 | es_chunk: True
Subject: Re: Todd & Things....
Body: The Astros play St. Louis on both of those 
days., both evening games. I would be happy to give you four tickets for 
either one of those games. Just ...

ID: 145 | Distancia

In [ ]:
resultado4 = collection_completos.query(
    query_texts=["Sally is out of the office"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado4["ids"][0], resultado4["documents"][0], resultado4["metadatas"][0], resultado4["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f} | es_chunk: {meta.get('es_chunk')}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

ID: 808 | Distancia: 0.3546 | es_chunk: False
Subject: Re: Manual Wires
Body: [2000-12-21 09:19:00+00:00] patti.thompson@enron.com:
Laurel,

Sally's plans are to be in the office Jan 2 & 5 only.  She will be out of 
town Jan 3 &...

ID: 807 | Distancia: 0.4202 | es_chunk: False
Subject: Re: ENW Staff Mtg. Thursday, 12/21/00
Body: [2000-12-20 08:44:00+00:00] patti.thompson@enron.com:
Tammie,

Sally is on vacation this week, therefore she will not be attending staff 
meeting.  Sh...

ID: 1052 | Distancia: 0.4393 | es_chunk: False
Subject: Re: VMS Project Next Steps
Body: [2001-03-15 11:00:00+00:00] patti.thompson@enron.com:
Murray,

Sally Beck and Leslie Reeves are both on vacation this week.  Sally will need 
to discu...

ID: 630 | Distancia: 0.4489 | es_chunk: False
Subject: RE: Wednesday
Body: [2000-11-20 18:01:00+00:00] sally.beck@enron.com:
I am out of the office all day on Wednesday. --Sally...

ID: 533 | Distancia: 0.4602 | es_chunk: False
Subject: Re: WorkLife Executive Council M

In [ ]:
resultado5 = collection_completos.query(
    query_texts=["Who won the 2026 World Cup?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado5["ids"][0], resultado5["documents"][0], resultado5["metadatas"][0], resultado5["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f} | es_chunk: {meta.get('es_chunk')}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

ID: 2169 | Distancia: 0.6913 | es_chunk: False
Subject: Documentation
Body: [2002-01-16 20:47:16+00:00] amy.fitzpatrick@enron.com:
Bridgette Anderson
Diane Anderson
Robert Bowen
Anthonly Campos
Sylvia Campos
Sharen Cason
Rhond...

ID: 1265 | Distancia: 0.7443 | es_chunk: False
Subject: RE: TeamBuilding for CommdodityLogic
Body: [2001-04-24 22:52:26+00:00] sally.beck@enron.com:
Thanks for the invitation.  I will be out of town April 25 through 27, so unfortunately I will have ...

ID: 1655 | Distancia: 0.7456 | es_chunk: False
Subject: FW: Power Settlement- Manuel Wires
Body: [2001-10-26 01:35:34+00:00] sally.beck@enron.com:
Let's talk tomorrow about how we get everyone on the ball and thinking....

ID: 1945_chunk11 | Distancia: 0.7458 | es_chunk: True
Subject: Enron Mentions- 11/27/01
Body: start of 2001....

ID: 1915_chunk43 | Distancia: 0.7549 | es_chunk: True
Subject: Enron Mentions -  11/23/01
Body: in the third quarter....



In [ ]:
resultado6 = collection_completos.query(
    query_texts=["What is the chemical symbol for gold?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado6["ids"][0], resultado6["documents"][0], resultado6["metadatas"][0], resultado6["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 54.2MiB/s]


ID: 566_chunk4 | Distancia: 0.7314
Subject: Enron Metals Update
Body: ? merchanting of recycled metal = 01 ) enron metals trades as a principal wit = h = 20 suppliers and consumers of non - ferrous metal for recycling. e...

ID: 1623_chunk5 | Distancia: 0.7436
Subject: National Web Hub info
Body: some names and numbers ) and a variety of forms....

ID: 566_chunk2 | Distancia: 0.7723
Subject: Enron Metals Update
Body: = 20 merchanting division the merchanting division, led by michael farmer ( + 44 ( 0 ) 20 7783 2049 ), is a = = 20 worldwide copper merchant, one of t...

ID: 821_chunk0 | Distancia: 0.7850
Subject: Re: ENA RMT Confirmations
Body: [2000-12-28 10:01:00+00:00] sally.beck@enron.com:
Kim, do you have an estimated completion date for the commodities where the 
notation is that inform...

ID: 1822 | Distancia: 0.7989
Subject: ERV Notification:  (DPR - Concentration - 11/12/2001)
Body: [2001-11-13 22:44:32+00:00] cassi.wallace@enron.com:
The report named: DPR - Concentration <htt

In [ ]:
resultado7 = collection_completos.query(
    query_texts=["Who directed the movie Inception?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado7["ids"][0], resultado7["documents"][0], resultado7["metadatas"][0], resultado7["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

ID: 1959 | Distancia: 0.7267
Subject: Pictures of Savannah
Body: [2001-12-17 18:26:03+00:00] kevin.sweeney@enron.com:
Attached are some early photos of the baby....

ID: 113 | Distancia: 0.7663
Subject: Organisational Announcement - Tani Nath
Body: [2000-03-09 09:28:00+00:00] sally.beck@enron.com:
FYI - This should make things easier in London as there will be one point 
person to focus on global...

ID: 389 | Distancia: 0.7670
Subject: Your approval is requested
Body: [2000-08-23 15:22:00+00:00] sally.beck@enron.com:
Do you know who this is?  I don't recognize the manager's name.  Thanks....

ID: 1883 | Distancia: 0.7693
Subject: RE: AEP HR
Body: [2001-11-20 17:12:06+00:00] sally.beck@enron.com:
Thanks - Anne did call me this morning with the details.  --Sally...

ID: 2093 | Distancia: 0.7703
Subject: Launch Plan
Body: [2002-01-10 14:45:44+00:00] david.forster@enron.com:
Is located in:

o:\eol\Newco Online\Pitch Book

There is a separate file for each chapter. An ove...



La indexación funciona correctamente.
La búsqueda semántica distingue bien entre "relacionado" y "la respuesta correcta" — la distancia coseno refleja esa diferencia de relevancia.
Tienes una base sólida para construir el banco de queries y comparar A/B/C más adelante.

##Indexación B

Conectamos con la API de Gemini para generar, mediante un agente resumidor, un resumen de cada uno de los 163 correos, creando así un segundo dataset con el resumen y los tokens consumidos en cada llamada.

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')
client_gemini = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.1-flash-lite"
print(f"✅ Conexión establecida con {MODEL_NAME}.")

✅ Conexión establecida con gemini-3.1-flash-lite.


###Pruebas

In [ ]:
import json
import time

def agente_resumidor_1(cuerpo_correo):
    prompt_resumen = f"""Resume el correo conservando personas, decisiones y fechas clave.
Si tiene menos de 15 palabras, cópialo tal cual como resumen.
Responde en el mismo idioma del correo.
CORREO:
{cuerpo_correo}
Devuelve solo JSON: {{"resumen": "..."}}"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_resumen,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.2,
        }
    )

    tiempo_ejecucion = time.time() - inicio

    resumen = json.loads(respuesta.text)["resumen"]

    return {
        "resumen": resumen,
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
    }

In [ ]:
print(df_beck_final["body"].iloc[0])

[1999-12-13 22:25:00+00:00] sally.beck@enron.com:
Please give Gretel a final list of dates that you plan to be out of the 
office from December 20 through the end of the year.  For the dates that you 
will be out, please provide the name and number of the key contact person in 
your group that will be in the office on the days that you are out.  

As I have in past years, I will compile a list of key contacts through the 
holiday period that we can use internally within Energy Operations.  I will 
also provide this list to others outside Energy Operations who may needa  
contact list during this time period.  

Please get this info to Gretel by Wednesday at noon, so that I have time to 
finalize the contact list before I am out on vacation.  

FYI:  I plan to be out of the office the week of December 20.  I will be at 
home during that week and can be reached there if necessary.  I will be back 
in the office on Monday, December 27 and I will be working through the end of 
the year.


In [ ]:
for i in range(5):
  ejemplo = df_beck_final["body"].iloc[i]
  resultado = agente_resumidor_1(ejemplo)
  print(resultado)

{'resumen': 'Sally Beck solicita a su equipo enviar a Gretel antes del miércoles al mediodía las fechas de vacaciones (del 20 al 31 de diciembre) y los contactos de respaldo para compilar una lista de emergencias.', 'prompt_tokens': 337, 'output_tokens': 57, 'total_tokens': 394, 'tiempo_ejecucion': 0.802}
{'resumen': 'Sally Beck confirmó el 15 de diciembre de 1999 que no habrá reunión de personal el lunes 20 de diciembre.', 'prompt_tokens': 134, 'output_tokens': 35, 'total_tokens': 169, 'tiempo_ejecucion': 0.612}
{'resumen': 'Sally Beck solicita una reunión con Jim Cole para la semana después de Navidad, tras no poder concretar una cita antes de sus vacaciones.', 'prompt_tokens': 260, 'output_tokens': 37, 'total_tokens': 297, 'tiempo_ejecucion': 0.805}
{'resumen': 'Sally Beck decidió el 1999-12-16 gestionar el asunto mediante reembolso de gastos.', 'prompt_tokens': 113, 'output_tokens': 29, 'total_tokens': 142, 'tiempo_ejecucion': 0.625}
{'resumen': 'Sally Beck declinó la invitación pa

In [ ]:
import json
import time

def agente_resumidor(cuerpo_correo):
    prompt_resumen = f"""Resume el correo conservando personas, decisiones y fechas clave.
Si tiene menos de 15 palabras, cópialo tal cual como resumen.
CORREO:
{cuerpo_correo}
Responde en el mismo idioma del correo.
Devuelve solo JSON: {{"resumen": "..."}}"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_resumen,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.2,
        }
    )

    tiempo_ejecucion = time.time() - inicio

    resumen = json.loads(respuesta.text)["resumen"]

    return {
        "resumen": resumen,
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
    }

In [ ]:
for i in range(5):
  print(df_beck_final["body"].iloc[i])
  print('************************')

[1999-12-13 22:25:00+00:00] sally.beck@enron.com:
Please give Gretel a final list of dates that you plan to be out of the 
office from December 20 through the end of the year.  For the dates that you 
will be out, please provide the name and number of the key contact person in 
your group that will be in the office on the days that you are out.  

As I have in past years, I will compile a list of key contacts through the 
holiday period that we can use internally within Energy Operations.  I will 
also provide this list to others outside Energy Operations who may needa  
contact list during this time period.  

Please get this info to Gretel by Wednesday at noon, so that I have time to 
finalize the contact list before I am out on vacation.  

FYI:  I plan to be out of the office the week of December 20.  I will be at 
home during that week and can be reached there if necessary.  I will be back 
in the office on Monday, December 27 and I will be working through the end of 
the year.
**

In [ ]:
for i in range(5):
  ejemplo = df_beck_final["body"].iloc[i]
  resultado = agente_resumidor(ejemplo)
  print(resultado)

{'resumen': 'Sally Beck requested Gretel to compile a holiday contact list by Wednesday at noon, covering absences from December 20 through year-end.', 'prompt_tokens': 337, 'output_tokens': 38, 'total_tokens': 375, 'tiempo_ejecucion': 0.698}
{'resumen': 'Sally Beck confirmed there is no staff meeting on Monday, December 20.', 'prompt_tokens': 134, 'output_tokens': 26, 'total_tokens': 160, 'tiempo_ejecucion': 0.538}
{'resumen': 'Sally Beck requested a meeting with Jim Cole for the week after Christmas, following her PRC on December 14.', 'prompt_tokens': 260, 'output_tokens': 33, 'total_tokens': 293, 'tiempo_ejecucion': 0.627}
{'resumen': 'On 1999-12-16, Sally Beck decided to handle the matter through expense reimbursement.', 'prompt_tokens': 113, 'output_tokens': 34, 'total_tokens': 147, 'tiempo_ejecucion': 0.624}
{'resumen': 'Sally Beck will not attend the meeting next week due to vacation (1999-12-16).', 'prompt_tokens': 131, 'output_tokens': 34, 'total_tokens': 165, 'tiempo_ejecuci

Provamos como funciona la huella de CO2 para API's

In [ ]:
!pip install "ecologits[google-genai]" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 684.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires google-genai<3,>=2.9, but you have google-genai 1.75.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [ ]:
from ecologits import EcoLogits

EcoLogits.init(providers=["google_genai"])

In [ ]:
def agente_resumidor(cuerpo_correo):
    prompt_resumen = f"""Resume el correo conservando personas, decisiones y fechas clave.
    Si tiene menos de 15 palabras, cópialo tal cual como resumen.

    CORREO:
    {cuerpo_correo}
    Responde en el mismo idioma del correo.

    Devuelve solo JSON: {{"resumen": "..."}}"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_resumen,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.2,
        }
    )

    tiempo_ejecucion = time.time() - inicio
    resumen = json.loads(respuesta.text)["resumen"]

    return {
        "resumen": resumen,
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
        "energia_kwh": respuesta.impacts.energy.value,
        "gwp_kgco2eq": respuesta.impacts.gwp.value,
    }

In [ ]:
ejemplo = df_beck_final["body"].iloc[0]
resultado = agente_resumidor(ejemplo)
print(resultado)

{'resumen': 'Sally Beck requests Gretel to compile a holiday contact list by Wednesday at noon, covering absences from December 20 through year-end.', 'prompt_tokens': 346, 'output_tokens': 38, 'total_tokens': 384, 'tiempo_ejecucion': 0.693, 'energia_kwh': RangeValue(min=2.8747878620978728e-06, max=1.857468840916571e-05), 'gwp_kgco2eq': RangeValue(min=1.2171123366774007e-06, max=7.588285754431213e-06)}


In [ ]:
def agente_resumidor(cuerpo_correo):
    prompt_resumen = f"""Resume el correo conservando personas, decisiones y fechas clave.
    Si tiene menos de 15 palabras, cópialo tal cual como resumen.

    CORREO:
    {cuerpo_correo}
    Responde en el mismo idioma del correo.

    Devuelve solo JSON: {{"resumen": "..."}}"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_resumen,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.2,
        }
    )

    tiempo_ejecucion = time.time() - inicio
    resumen = json.loads(respuesta.text)["resumen"]

    energia = respuesta.impacts.energy.value  # RangeValue en kWh
    gwp = respuesta.impacts.gwp.value          # RangeValue en kgCO2eq

    return {
        "resumen": resumen,
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
        "energia_wh": round(energia.mean * 1000, 4),
        "gwp_mgco2eq": round(gwp.mean * 1_000_000, 4),
    }

In [ ]:
ejemplo = df_beck_final["body"].iloc[0]
resultado = agente_resumidor(ejemplo)
print(resultado)

{'resumen': 'Sally Beck requested Gretel to compile a holiday contact list by Wednesday at noon, covering absences from December 20 through year-end. Sally will be out the week of December 20 and returns December 27.', 'prompt_tokens': 346, 'output_tokens': 56, 'total_tokens': 402, 'tiempo_ejecucion': 0.735, 'energia_wh': 0.0153, 'gwp_mgco2eq': 6.1745}


Calcular tiempo ejecución

Code Carbon Libreria

https://pypi.org/project/codecarbon/

###Resumenes

In [ ]:
!pip install "ecologits[google-genai]" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 782.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.0 requires google-genai<3,>=2.12.1, but you have google-genai 1.75.0 which is incompatible.
google-adk 2.7.0 requires opentelemetry-api<=1.43,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.0 requires opentelemetry-sdk<=1.43,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [ ]:
from ecologits import EcoLogits

EcoLogits.init(providers=["google_genai"])

In [ ]:
def agente_resumidor(cuerpo_correo):
    prompt_resumen = f"""Resume el correo conservando personas, decisiones y fechas clave.
    Si tiene menos de 15 palabras, cópialo tal cual como resumen.

    CORREO:
    {cuerpo_correo}
    Responde en el mismo idioma del correo.

    Devuelve solo JSON: {{"resumen": "..."}}"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_resumen,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.2,
        }
    )

    tiempo_ejecucion = time.time() - inicio
    resumen = json.loads(respuesta.text)["resumen"]

    energia = respuesta.impacts.energy.value  # RangeValue en kWh
    gwp = respuesta.impacts.gwp.value          # RangeValue en kgCO2eq

    return {
        "resumen": resumen,
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
        "energia_wh": round(energia.mean * 1000, 4),
        "gwp_mgco2eq": round(gwp.mean * 1_000_000, 4),
    }

In [ ]:
df_resumenes = df_beck_final.copy()
resultados = []

In [ ]:
df_resumenes = df_beck_final.copy()
resultados = pd.read_csv("/content/drive/MyDrive/TFM/resultados_parcial.csv").to_dict("records")
print(len(resultados))

1800


In [ ]:
tamano_tanda = 100
numero_tanda = 22

inicio = (numero_tanda - 1) * tamano_tanda
fin = inicio + tamano_tanda

for i, correo in enumerate(df_resumenes["body"].iloc[inicio:fin]):
    print(f"Procesando índice {inicio + i}...")
    resultado = agente_resumidor(correo)
    resultados.append(resultado)
    time.sleep(4)

pd.DataFrame(resultados).to_csv("/content/drive/MyDrive/TFM/resultados_parcial.csv", index=False)

print(f"Tanda {numero_tanda} completada. Total acumulado: {len(resultados)}/{len(df_resumenes)}")

Procesando índice 2100...
Procesando índice 2101...
Procesando índice 2102...
Procesando índice 2103...
Procesando índice 2104...
Procesando índice 2105...
Procesando índice 2106...
Procesando índice 2107...
Procesando índice 2108...
Procesando índice 2109...
Procesando índice 2110...
Procesando índice 2111...
Procesando índice 2112...
Procesando índice 2113...
Procesando índice 2114...
Procesando índice 2115...
Procesando índice 2116...
Procesando índice 2117...
Procesando índice 2118...
Procesando índice 2119...
Procesando índice 2120...
Procesando índice 2121...
Procesando índice 2122...
Procesando índice 2123...
Procesando índice 2124...
Procesando índice 2125...
Procesando índice 2126...
Procesando índice 2127...
Procesando índice 2128...
Procesando índice 2129...
Procesando índice 2130...
Procesando índice 2131...
Procesando índice 2132...
Procesando índice 2133...
Procesando índice 2134...
Procesando índice 2135...
Procesando índice 2136...
Procesando índice 2137...
Procesando í

In [ ]:
for i, correo in enumerate(df_resumenes["body"].iloc[2200:2229]):
    print(f"Procesando índice {2200 + i}...")
    resultado = agente_resumidor(correo)
    resultados.append(resultado)
    time.sleep(4)

pd.DataFrame(resultados).to_csv("/content/drive/MyDrive/TFM/resultados_parcial.csv", index=False)
print(f"Total acumulado: {len(resultados)}/{len(df_resumenes)}")

Procesando índice 2200...
Procesando índice 2201...
Procesando índice 2202...
Procesando índice 2203...
Procesando índice 2204...
Procesando índice 2205...
Procesando índice 2206...
Procesando índice 2207...
Procesando índice 2208...
Procesando índice 2209...
Procesando índice 2210...
Procesando índice 2211...
Procesando índice 2212...
Procesando índice 2213...
Procesando índice 2214...
Procesando índice 2215...
Procesando índice 2216...
Procesando índice 2217...
Procesando índice 2218...
Procesando índice 2219...
Procesando índice 2220...
Procesando índice 2221...
Procesando índice 2222...
Procesando índice 2223...
Procesando índice 2224...
Procesando índice 2225...
Procesando índice 2226...
Procesando índice 2227...
Procesando índice 2228...
Total acumulado: 2229/2229


In [ ]:
print(len(resultados))
#pd.DataFrame(resultados).to_csv("/content/drive/MyDrive/TFM/resultados_parcial.csv", index=False)

1800


In [ ]:
verificacion = pd.read_csv("/content/drive/MyDrive/TFM/resultados_parcial.csv")
print(len(verificacion))

2229


In [ ]:
verificacion.iloc[1860:1864]

,resumen,prompt_tokens,output_tokens,total_tokens,tiempo_ejecucion,energia_wh,gwp_mgco2eq
1860,"On 2001-11-19, Mark Pickering asked Sally abou...",241,56,297,0.751,0.0153,6.1999
1861,first that would help you. Sorry should have ...,121,23,144,0.452,0.0066,2.7018
1862,"On 2001-11-19, Mark Pickering asked Sally if G...",157,42,199,0.677,0.0117,4.7739
1863,Kenneth Thibodeaux announced that the 11/16/01...,156,37,193,0.531,0.0102,4.1344


In [ ]:
verificacion.tail(3)

,resumen,prompt_tokens,output_tokens,total_tokens,tiempo_ejecucion,energia_wh,gwp_mgco2eq
2226,"On 2002-01-27, Tim Belden added Scotty Gilbert...",150,48,198,0.477,0.0128,5.1331
2227,"On 2002-01-28, Sally Beck requested coordinati...",171,44,215,0.492,0.0119,4.7643
2228,"On 2002-01-28, Sally Beck informed that Paige ...",150,43,193,0.574,0.0118,4.7579


In [ ]:
import random

df_resumenes["resumen"] = [r["resumen"] for r in resultados]

indices_prueba = [0, 400, 497, 900, 1397, 1861, 1900, 2200, 2228] + random.sample(range(len(df_resumenes)), 5)

for i in sorted(set(indices_prueba)):
    print(f"=== Índice {i} ===")
    print("ORIGINAL:", df_resumenes["body"].iloc[i])
    print("RESUMEN :", df_resumenes["resumen"].iloc[i])
    print()

=== Índice 0 ===
ORIGINAL: [1999-12-13 22:25:00+00:00] sally.beck@enron.com:
Please give Gretel a final list of dates that you plan to be out of the 
office from December 20 through the end of the year.  For the dates that you 
will be out, please provide the name and number of the key contact person in 
your group that will be in the office on the days that you are out.  

As I have in past years, I will compile a list of key contacts through the 
holiday period that we can use internally within Energy Operations.  I will 
also provide this list to others outside Energy Operations who may needa  
contact list during this time period.  

Please get this info to Gretel by Wednesday at noon, so that I have time to 
finalize the contact list before I am out on vacation.  

FYI:  I plan to be out of the office the week of December 20.  I will be at 
home during that week and can be reached there if necessary.  I will be back 
in the office on Monday, December 27 and I will be working throu

###Indexación

In [ ]:
df_beck_final = pd.read_csv("/content/drive/MyDrive/TFM/dataset_beck_final.csv")
print(len(df_beck_final))

In [ ]:
print(len(df_beck_final))

2229


In [ ]:
df_resumens= pd.read_csv("/content/drive/MyDrive/TFM/resultados_parcial.csv")

In [ ]:
print(len(df_resumens))

2229


In [ ]:
df_beck_final.head()

,subject,from_addr,fecha_primer_mensaje,fecha_ultimo_mensaje,body,n_mensajes
0,Key Contacts in Your Area - December 20 - 31,sally.beck@enron.com,1999-12-13 22:25:00+00:00,1999-12-13 22:25:00+00:00,[1999-12-13 22:25:00+00:00] sally.beck@enron.c...,1
1,Re: Key Contacts in Your Area - December 20 - 31,sally.beck@enron.com,1999-12-15 14:14:00+00:00,1999-12-15 14:14:00+00:00,[1999-12-15 14:14:00+00:00] sally.beck@enron.c...,1
2,Re: Rotations,sally.beck@enron.com,1999-12-13 20:45:00+00:00,1999-12-16 08:42:00+00:00,[1999-12-13 20:45:00+00:00] sally.beck@enron.c...,2
3,Re: CFA Exam and Seminar,sally.beck@enron.com,1999-12-16 10:39:00+00:00,1999-12-16 10:39:00+00:00,[1999-12-16 10:39:00+00:00] sally.beck@enron.c...,1
4,Re: Project Aruba Closing Dinner,sally.beck@enron.com,1999-12-16 14:12:00+00:00,1999-12-16 14:12:00+00:00,[1999-12-16 14:12:00+00:00] sally.beck@enron.c...,1


In [ ]:
df_resumens.head()

,resumen,prompt_tokens,output_tokens,total_tokens,tiempo_ejecucion,energia_wh,gwp_mgco2eq
0,Sally Beck requested Gretel to compile a holid...,342,38,380,1.086,0.0114,4.8321
1,Sally Beck confirmed no staff meeting on Monda...,139,24,163,0.837,0.0075,3.2176
2,Sally Beck requested a meeting with Jim Cole f...,265,33,298,0.628,0.0094,3.8549
3,"On 1999-12-16, Sally Beck decided to handle th...",118,34,152,0.628,0.0096,3.9506
4,Sally Beck declined the invitation for next we...,136,34,170,0.799,0.0099,4.1372


In [ ]:
df_resumenes = pd.concat([df_beck_final, df_resumens], axis=1)
print(df_resumenes.shape)
print(df_resumenes.columns.tolist())

(2229, 13)
['subject', 'from_addr', 'fecha_primer_mensaje', 'fecha_ultimo_mensaje', 'body', 'n_mensajes', 'resumen', 'prompt_tokens', 'output_tokens', 'total_tokens', 'tiempo_ejecucion', 'energia_wh', 'gwp_mgco2eq']


In [ ]:
!pip install transformers -q

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
n_tokens_resumen = df_resumenes["resumen"].apply(lambda x: len(tokenizer.encode(x, truncation=False)))

print(n_tokens_resumen.describe())
print()
truncados_resumen = (n_tokens_resumen > 256).sum()
print(f"Resúmenes que superan 256 tokens: {truncados_resumen} de {len(df_resumenes)} ({truncados_resumen/len(df_resumenes)*100:.1f}%)")

count    2229.000000
mean       32.206371
std        14.860015
min         3.000000
25%        23.000000
50%        29.000000
75%        38.000000
max       138.000000
Name: resumen, dtype: float64

Resúmenes que superan 256 tokens: 0 de 2229 (0.0%)


In [ ]:
import chromadb

cliente_resumenes = chromadb.PersistentClient(path="/content/drive/MyDrive/TFM/chroma_beck_B")
collection_resumenes = cliente_resumenes.get_or_create_collection(
    name="beck_resumenes",
    metadata={"hnsw:space": "cosine"}
)

print(collection_resumenes.count())

2229


In [ ]:
documents = df_resumenes["resumen"].tolist()
ids = [str(i) for i in df_resumenes.index]
metadatas = df_resumenes[["subject", "from_addr", "fecha_primer_mensaje", "fecha_ultimo_mensaje", "n_mensajes"]].astype(str).to_dict("records")

print(len(documents))

2229


In [ ]:
import time

inicio_indexacion_b = time.time()

collection_resumenes.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

tiempo_indexacion_b = time.time() - inicio_indexacion_b

print(collection_resumenes.count())
print(f"Tiempo de indexación: {tiempo_indexacion_b:.2f} segundos")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 37.4MiB/s]


2229
Tiempo de indexación: 409.33 segundos


###Comprobaciones

In [ ]:
resultado_b1 = collection_resumenes.query(
    query_texts=["Who was appointed as manager of the West region?"],
    n_results=3
)

for id_doc, doc, meta, dist in zip(resultado_b1["ids"][0], resultado_b1["documents"][0], resultado_b1["metadatas"][0], resultado_b1["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Resumen: {doc}")
    print()

ID: 162 | Distancia: 0.4223
Subject: Re: Reorganization/Promotion
Resumen: On 2000-04-18, Sally Beck, following Susan and Brent's recommendation, appointed Jeff Gossett as West region manager, eliminating OA and Deal Validation roles.

ID: 1466 | Distancia: 0.6552
Subject: Reminder - Final PRC - Friday, June 8
Resumen: Patti Thompson scheduled the Global Risk Management Operations Mid-Year PRC Meeting for June 8, 2001, at the DoubleTree Hotel. Sally Beck, Wes Colwell, and Greg Piper will join at 10:00 AM.

ID: 1648 | Distancia: 0.6604
Subject: Natural Gas Origination
Resumen: On 2001-10-25, Louise and John announced organizational changes: Fred Lagrasta will lead the consolidated Producer/Wellhead Group, Laura Luce will head Strategic Asset Development for Central/Eastern regions, and Frank Vickers will lead Central Origination and Mid-Market.



In [ ]:
resultado_b2 = collection_resumenes.query(
    query_texts=["Who did Sally ask to assess the impact of losing Bob?"],
    n_results=3
)

for id_doc, doc, meta, dist in zip(resultado_b2["ids"][0], resultado_b2["documents"][0], resultado_b2["metadatas"][0], resultado_b2["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Resumen: {doc}")
    print()

ID: 1075 | Distancia: 0.3725
Subject: Re: Material for Commodity Logic meeting
Resumen: Patti Thompson informed Bob that Sally is out until Thursday at 8:00 for a CommodityLogic meeting; Bob Hall will handle the matter.

ID: 343 | Distancia: 0.4025
Subject: Re: United Way Kick Off
Resumen: On 2000-08-09, Sally Beck asked to arrange something with Bobbi.

ID: 1221 | Distancia: 0.4166
Subject: Re: Shortening of GL Close
Resumen: On 2001-04-18, Sally Beck requested to be kept informed about the impact of losing that day.



Este es un resultado importante y negativo que hay que documentar con cuidado: el correo correcto (ID 254, "Re: Personnel for EBS") no aparece en absoluto entre los 3 primeros resultados de la Configuración B — a diferencia de la Configuración A, donde sí aparecía en primer lugar con distancia 0,5393.

Además, hay un caso curioso: el resumen del ID 1221 dice "Sally Beck requested to be kept informed about the impact of losing that day" — esto suena como si el resumen hubiera perdido el sujeto real ("losing Bob" se convirtió en algo ambiguo como "losing that day"), posiblemente porque el resumen comprimió demasiado el contexto original y ahora resulta confuso incluso para un lector humano, no solo para el modelo de búsqueda.

Esto es exactamente el tipo de hallazgo que justifica tu comparación A/B/C: el resumen, en este caso concreto, perdió información clave (quién es "Bob", qué significa "losing" en ese contexto) que sí estaba disponible en el correo completo, y como consecuencia la búsqueda semántica no logró encontrar el documento correcto.

In [ ]:
print("Subject:", df_resumenes.loc[254, "subject"])
print()
print("BODY ORIGINAL:")
print(df_resumenes.loc[254, "body"])
print()
print("RESUMEN GENERADO:")
print(df_resumenes.loc[254, "resumen"])

Subject: Re: Personnel for EBS

BODY ORIGINAL:
[2000-06-12 17:41:00+00:00] sally.beck@enron.com:
I am not close enough to know Bob's exact role within the group to know the 
impact of the prospect of losing him.  I am forwarding this to Brent to get 
his assessment of the impact.  I do know that the number of deals that we do 
currently has skyrocketed with EOL, so I feel sure that the group is 
stretched right now.  Again, though, Brent would best be able to address 
this.  As you are probably aware, I personnally made the commitment to Bob 
several years ago to help in managing his workload so that he is available as 
needed at home for his wife.  For Bob's sake, I would want to make sure that 
he is in a situation where that is understood and appreciated.  

Thank you for contacting me first before going to Bob.  --Sally

RESUMEN GENERADO:
On 2000-06-12, Sally Beck forwarded to Brent an assessment regarding Bob's potential departure, noting the group's high workload and her prior co

In [ ]:
resultado_amplio = collection_resumenes.query(
    query_texts=["Who did Sally ask to assess the impact of losing Bob?"],
    n_results=20
)

if "254" in resultado_amplio["ids"][0]:
    posicion = resultado_amplio["ids"][0].index("254")
    print(f"El documento 254 está en la posición {posicion + 1} de 20")
    print(f"Distancia: {resultado_amplio['distances'][0][posicion]:.4f}")
else:
    print("No aparece ni en el top 20")

El documento 254 está en la posición 4 de 20
Distancia: 0.4333


In [ ]:
resultado_b3 = collection_resumenes.query(
    query_texts=["How many Astros tickets did Sally receive for the exhibition game on April 1?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado_b3["ids"][0], resultado_b3["documents"][0], resultado_b3["metadatas"][0], resultado_b3["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Resumen: {doc}")
    print()

ID: 137 | Distancia: 0.2486
Subject: Astros Tickets
Resumen: On 2000-03-29, Sally Beck announced the distribution of Astros season tickets among direct reports. Additionally, she allocated exhibition tickets (April 1) from Cliff Baxter, Greg Whalley, and Dave Delainey to other managers and directors, and planned future group outings.

ID: 1192 | Distancia: 0.3264
Subject: Astros Tickets 2001 Season
Resumen: On 2001-04-10, Sally Beck announced that Patti will distribute 2001 Astros tickets to teams proportionally, featuring new Dugout and Terrace level seats.

ID: 116 | Distancia: 0.3271
Subject: Re: Todd & Things....
Resumen: On 2000-03-13, Sally Beck confirmed ENA covers Todd's family travel costs. Sally will visit London on March 20th and offered Astros tickets for April 10th or 12th.

ID: 1183 | Distancia: 0.3302
Subject: Astros Tickets 2001 Season
Resumen: On 2001-04-09, Sally Beck announced that Patti will distribute 2001 Astros tickets to teams proportionally, featuring new, impr

Excelente resultado — el correo correcto (ID 137) queda en primer lugar, con distancia 0,2486, mejor incluso que la Configuración A con chunking (0,3668) para esta misma pregunta. Aquí el resumen funciona muy bien: al condensar directamente "exhibition tickets (April 1) from Cliff Baxter, Greg Whalley, and Dave Delainey", captura justo la información específica de la pregunta sin necesidad de fragmentar nada.

Con esto ya tienes una buena batería de comparaciones A vs B:

Jeff Gossett: prácticamente igual (0,41 vs 0,42).
Bob/Brent: A mejor (top-1, 0,54) vs B peor (posición 4, 0,43).
Astros Tickets: B mejor (0,25) vs A con chunking (0,37).

In [ ]:
resultado_b4 = collection_resumenes.query(
    query_texts=["Sally is out of the office"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado_b4["ids"][0], resultado_b4["documents"][0], resultado_b4["metadatas"][0], resultado_b4["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Resumen: {doc}")
    print()

ID: 630 | Distancia: 0.2403
Subject: RE: Wednesday
Resumen: Sally Beck will be out of the office all day on Wednesday, November 22, 2000.

ID: 631 | Distancia: 0.2446
Subject: RE: Teambuilding 11/21
Resumen: Sally Beck will be out of the office on Tuesday, November 21, 2000.

ID: 1033 | Distancia: 0.2619
Subject: Re: Wellesley Interview reschedule dates
Resumen: Sally Beck will be out of the office March 12-16 for spring break.

ID: 1017 | Distancia: 0.2804
Subject: Out of the Office - March 2 and March 5
Resumen: Sally Beck will be out of office March 2 and 5, returning March 6. Patti Thompson will assist.

ID: 251 | Distancia: 0.2887
Subject: Out of the Office - Afternoon of Friday, June 9
Resumen: Sally Beck will be out of the office after noon on Friday, June 9, and will not attend the afternoon meeting.



Muy buen resultado, y con una diferencia clara respecto a la Configuración A: aquí las distancias son notablemente más bajas (0,24-0,29, todas por debajo de las de A: 0,35-0,46), y todos los 5 resultados son casos genuinos y muy consistentes de "estar fuera de la oficina", con fechas y motivos concretos claramente indicados por el resumen.

Esto sugiere que, para preguntas genéricas de este tipo, los resúmenes producen coincidencias semánticas más ajustadas que los correos completos

In [ ]:
resultado_b5 = collection_resumenes.query(
    query_texts=["Who won the 2026 World Cup?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado_b5["ids"][0], resultado_b5["documents"][0], resultado_b5["metadatas"][0], resultado_b5["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Resumen: {doc}")
    print()

ID: 1024 | Distancia: 0.6979
Subject: Report from State Competition
Resumen: On 2001-03-06, Sally Beck reported that the Klein Bearkadettes won multiple awards, including grand champion, at a Dallas dance competition.

ID: 2163 | Distancia: 0.7039
Subject: Texas Tribute Online -- January 2002
Resumen: In January 2002, Ben Crenshaw and Justin Leonard donated Ryder Cup winnings to UT's golf program. Photographer George O. Jackson donated his 'Essence of Mexico' project to the Benson Collection, to be archived over two years. Additionally, UT promoted the 2002 StarDate Sky Almanac to support the McDonald Observatory.

ID: 148 | Distancia: 0.7343
Subject: Re: Enron MBA Excellence Awards - Dinner
Resumen: On April 5-6, 2000, Sally Beck, Gene Humphrey, Thomas Suffield, and Roger Ondrecko selected three scholarship winners. Sally is coordinating with Sue White to finalize the notification process and planning a dinner in Austin for the week of April 24th, with potential attendance by Rick Cau

In [ ]:
resultado_b6 = collection_resumenes.query(
    query_texts=["What is the chemical symbol for gold?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado_b6["ids"][0], resultado_b6["documents"][0], resultado_b6["metadatas"][0], resultado_b6["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

ID: 1153 | Distancia: 0.7642
Subject: Peoples presentation
Body: Please print in color for me. Thanks....

ID: 1161 | Distancia: 0.7642
Subject: Peoples presentation
Body: Please print in color for me.  Thanks....

ID: 25 | Distancia: 0.7807
Subject: Vacation, floating holiday request
Body: Reference for future months....

ID: 235 | Distancia: 0.7827
Subject: Thunderball presentation
Body: Please pring attached presentation for me in color. Thanks....

ID: 2113 | Distancia: 0.7923
Subject: FW: Commodity Logic
Body: Any information would be helpful....



In [ ]:
resultado_b7 = collection_resumenes.query(
    query_texts=["Who directed the movie Inception?"],
    n_results=5
)

for id_doc, doc, meta, dist in zip(resultado_b7["ids"][0], resultado_b7["documents"][0], resultado_b7["metadatas"][0], resultado_b7["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(f"Body: {doc[:150]}...")
    print()

ID: 1959 | Distancia: 0.6775
Subject: Pictures of Savannah
Body: [2001-12-17] kevin.sweeney@enron.com shared early baby photos....

ID: 2226 | Distancia: 0.7544
Subject: Portland vm
Body: On 2002-01-27, Tim Belden added Scotty Gilbert, Eugene Lee, Donna Sheidun, and Heather Dunton to a previous Friday discussion....

ID: 2211 | Distancia: 0.7680
Subject: Cooperation with the FBI
Body: On 2002-01-24, Enron allowed the FBI to search its building and interview employees. The company retained Michael Levy to provide legal counsel, with ...

ID: 1112 | Distancia: 0.7694
Subject: Desktop Video Conferening Demo
Body: Patti Thompson announced that Sally arranged a demo of Avistar's video conferencing product by Brian Wood on April 4, 1:00-2:00 PM....

ID: 1784 | Distancia: 0.7747
Subject: IMPORTANT VIDEO ANNOUNCEMENT
Body: On 2001-11-09, chairman.ken@enron.com announced a video regarding the company's future....



Buen resultado de control — las distancias siguen siendo altas (0,70-0,74), consistentes con la Configuración A (0,69-0,78), confirmando que ningún resultado tiene relación real con la pregunta sobre el Mundial de fútbol. El sistema sigue comportándose correctamente ante preguntas sin respuesta en el dataset, tanto con resúmenes como con correos completos.

Con esto ya tienes las 5 comparaciones completas entre A y B. Resumen de hallazgos para documentar:

Pregunta	Config A (mejor resultado)	Config B (mejor resultado)
Jeff Gossett	0,4117 (top-1)	0,4223 (top-1) — prácticamente igual
Bob/Brent	0,5393 (top-1)	0,4333 (posición 4) — A mejor
Astros Tickets	0,3668 (top-1)	0,2486 (top-1) — B mejor
Out of office (genérico)	0,35-0,46	0,24-0,29 — B con distancias más bajas
World Cup (control)	0,70-0,78	0,70-0,74 — comportamiento equivalente

#Queries

In [ ]:
# --- Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_beck_final = pd.read_csv("/content/drive/MyDrive/TFM/dataset_beck_final.csv")
print(len(df_beck_final))

2229


In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')
client_gemini = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.1-flash-lite"
print(f"✅ Conexión establecida con {MODEL_NAME}.")

✅ Conexión establecida con gemini-3.1-flash-lite.


In [ ]:
!pip install transformers -q

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

n_tokens = df_beck_final["body"].apply(lambda x: len(tokenizer.encode(x, truncation=False)))

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
def agente_generador_preguntas(lote_correos):
    partes_texto = []
    for c in lote_correos:
        partes_texto.append(f"[CORREO {c['indice']}]\n{c['body']}")

    correos_texto = "\n\n".join(partes_texto)

    prompt = f"""Eres un asistente que diseña preguntas de evaluación para un sistema de asistencia de correo electrónico.

Los correos que se te muestran pertenecen a Sally Beck, quien usará este sistema como su asistente personal de correo. Genera preguntas como si las hiciera ella misma sobre su propio correo.

Se te dan varios correos numerados. Para cada correo que tenga contenido concreto y verificable, genera UNA pregunta en inglés junto con su respuesta correcta como si fueras Sally. La pregunta debe poder entenderse por sí sola, sin depender de saber previamente de qué correo o tema se trata: incluye en la propia pregunta el nombre, evento, fecha o asunto concreto al que se refiere, en lugar de usar referencias vagas como "the meeting mentioned in my email" o "the location I referred to".

Si un correo es demasiado corto, sin contenido relevante o muy general, como saludos o confirmaciones triviales, NO generes pregunta para él.

CORREOS:
{correos_texto}

Devuelve EXCLUSIVAMENTE un JSON con esta estructura:
{{
    "preguntas": [
        {{"indice": <número de correo>, "pregunta": "...", "respuesta": "..."}}
    ]
}}
"""

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.2,
        }
    )

    return json.loads(respuesta.text)["preguntas"]

count     2229.000000

mean       212.447734

std        870.931814

min         27.000000

25%         60.000000

50%        106.000000

75%        199.000000

max      25058.000000

In [ ]:
correos_cortos = df_beck_final[n_tokens <= 60].sample(frac=1, random_state=42)
muestra_prueba = correos_cortos.iloc[20:30]

lote_correos = []
for idx, fila in muestra_prueba.iterrows():
    lote_correos.append({"indice": idx, "body": fila["body"]})

resultado = agente_generador_preguntas(lote_correos)
print(f"Preguntas generadas: {len(resultado)} de {len(lote_correos)} correos")
for item in resultado:
    print(item)
    print()

Preguntas generadas: 2 de 10 correos
{'indice': 903, 'pregunta': 'When is Beth Apollo starting her new role in Houston and what is the purpose of my inquiry regarding her placement?', 'respuesta': "Beth Apollo is starting on January 29th, and I am looking for a workspace or 'room at the inn' for her to work with me."}

{'indice': 907, 'pregunta': 'Why am I unable to attend the meeting or event scheduled for this week?', 'respuesta': 'I am unable to attend because I am currently working from the London office.'}



In [ ]:
print(df_beck_final.loc[907, "body"])

[2001-01-17 08:07:00+00:00] sally.beck@enron.com:
I am in the London office this week, so I won't be able to attend.


In [ ]:
correos_largos = df_beck_final[n_tokens > 200].sample(frac=1, random_state=42)
muestra_largos_prueba2 = correos_largos.iloc[10:20]

lote_largos2 = []
for idx, fila in muestra_largos_prueba2.iterrows():
    lote_largos2.append({"indice": idx, "body": fila["body"]})

resultado_largos2 = agente_generador_preguntas(lote_largos2)
print(f"Preguntas generadas: {len(resultado_largos2)} de {len(lote_largos2)} correos")
for item in resultado_largos2:
    print(item)
    print()

Preguntas generadas: 9 de 10 correos
{'indice': 819, 'pregunta': 'What are my tentative travel plans for the London office visit in January 2001?', 'respuesta': 'I tentatively planned to fly to London on Sunday, January 14, 2001, to spend Monday through Thursday, January 15-18, in the office.'}

{'indice': 102, 'pregunta': 'What are the primary goals for Global Risk Management Operations for the year 2000?', 'respuesta': 'The goals are to build and enhance operational and controls infrastructure to support merchant activities, emerging businesses, and joint ventures, as well as to strengthen mid and back office talent and controls.'}

{'indice': 385, 'pregunta': 'What issues am I experiencing with the system that Beth Perlman mentioned I am in charge of?', 'respuesta': 'The system is not working correctly, has bugs, and was apparently not tested before being put into production.'}

{'indice': 995, 'pregunta': 'What are my vacation dates in March 2001 and what is the reason for them?', 

In [ ]:
pregunta ='What is the purpose of the working sessions between the ISC Steering Committee and Enron Networks (ENW)?'

resultado = collection_completos.query(query_texts=[pregunta], n_results=5)
for id_doc, dist in zip(resultado["ids"][0], resultado["distances"][0]):
    print(f"ID: {id_doc} | Distancia: {dist:.4f}")

print()
print("ID esperado: 819")

ID: 1836_chunk0 | Distancia: 0.2250
ID: 2179_chunk0 | Distancia: 0.3307
ID: 1836_chunk1 | Distancia: 0.3362
ID: 1470 | Distancia: 0.3601
ID: 1879 | Distancia: 0.3671

ID esperado: 819


##Queries Cortos

In [ ]:
correos_cortos = df_beck_final[n_tokens <= 60].sample(frac=1, random_state=42)  # baraja todos, sin repetir

preguntas_cortas = []
objetivo = 50
tamano_lote = 10
posicion = 0

while len(preguntas_cortas) < objetivo and posicion < len(correos_cortos):
    lote = correos_cortos.iloc[posicion:posicion + tamano_lote]

    lote_correos = []
    for idx, fila in lote.iterrows():
        lote_correos.append({"indice": idx, "body": fila["body"]})

    resultado_lote = agente_generador_preguntas(lote_correos)
    preguntas_cortas.extend(resultado_lote)

    posicion += tamano_lote
    print(f"Lote procesado (correos {posicion - tamano_lote}-{posicion}). Total acumulado: {len(preguntas_cortas)}/{objetivo}")

    time.sleep(4)

print("Terminado")

Lote procesado (correos 0-10). Total acumulado: 2/50
Lote procesado (correos 10-20). Total acumulado: 5/50
Lote procesado (correos 20-30). Total acumulado: 7/50
Lote procesado (correos 30-40). Total acumulado: 10/50
Lote procesado (correos 40-50). Total acumulado: 14/50
Lote procesado (correos 50-60). Total acumulado: 17/50
Lote procesado (correos 60-70). Total acumulado: 21/50
Lote procesado (correos 70-80). Total acumulado: 27/50
Lote procesado (correos 80-90). Total acumulado: 31/50
Lote procesado (correos 90-100). Total acumulado: 38/50
Lote procesado (correos 100-110). Total acumulado: 42/50
Lote procesado (correos 110-120). Total acumulado: 46/50
Lote procesado (correos 120-130). Total acumulado: 49/50
Lote procesado (correos 130-140). Total acumulado: 52/50
Terminado


In [ ]:
preguntas_cortas = preguntas_cortas[:50]
print(len(preguntas_cortas))

50


In [ ]:
for item in preguntas_cortas[10:16]:
    print(item)
    print()

{'indice': 1173, 'pregunta': 'Is the event scheduled for May 10 a lunch or a dinner?', 'respuesta': 'The email asks whether the event on May 10 is a lunch or a dinner.'}

{'indice': 1231, 'pregunta': 'Which rental car company should I use for my trip to Oklahoma?', 'respuesta': 'National is the rental car company to use in Oklahoma.'}

{'indice': 1265, 'pregunta': 'Why am I unable to attend the game scheduled between April 25 and April 27?', 'respuesta': 'I am unable to attend the game because I will be out of town from April 25 through April 27.'}

{'indice': 56, 'pregunta': 'What will be my primary focus while looking at operations around the world?', 'respuesta': 'My primary focus will be on ENA.'}

{'indice': 414, 'pregunta': 'What arrangement did I confirm for October 20th regarding the guest room?', 'respuesta': 'I confirmed that we will reserve the guest room for the guest.'}

{'indice': 2130, 'pregunta': 'What did Shona Wilson ask me regarding the gas books on the list?', 'resp

##Queries Intermedios

In [ ]:
correos_intermedios = df_beck_final[(n_tokens >= 60) & (n_tokens <= 200)].sample(frac=1, random_state=42)

preguntas_intermedias = []
objetivo = 50
tamano_lote = 10
posicion = 0

while len(preguntas_intermedias) < objetivo and posicion < len(correos_intermedios):
    lote = correos_intermedios.iloc[posicion:posicion + tamano_lote]

    lote_correos = []
    for idx, fila in lote.iterrows():
        lote_correos.append({"indice": idx, "body": fila["body"]})

    resultado_lote = agente_generador_preguntas(lote_correos)
    preguntas_intermedias.extend(resultado_lote)

    posicion += tamano_lote
    print(f"Lote procesado (correos {posicion - tamano_lote}-{posicion}). Total acumulado: {len(preguntas_intermedias)}/{objetivo}")

    time.sleep(4)

print("Terminado")

Lote procesado (correos 0-10). Total acumulado: 6/50
Lote procesado (correos 10-20). Total acumulado: 15/50
Lote procesado (correos 20-30). Total acumulado: 22/50
Lote procesado (correos 30-40). Total acumulado: 27/50
Lote procesado (correos 40-50). Total acumulado: 34/50
Lote procesado (correos 50-60). Total acumulado: 41/50
Lote procesado (correos 60-70). Total acumulado: 49/50
Lote procesado (correos 70-80). Total acumulado: 57/50
Terminado


In [ ]:
preguntas_intermedias = preguntas_intermedias[:50]
print(len(preguntas_intermedias))

50


In [ ]:
for item in preguntas_intermedias[10:16]:
    print(item)
    print()

{'indice': 1845, 'pregunta': 'Who will be covering for Lisa Costello and Mark Pickering while Lisa is on maternity leave starting November 19, 2001?', 'respuesta': 'Katherine Chisley will be covering for both Lisa Costello and Mark Pickering.'}

{'indice': 1411, 'pregunta': 'What is the rescheduled date and time for the EA Staff Meeting that was moved due to the Memorial Holiday?', 'respuesta': 'The EA Staff Meeting is rescheduled for Thursday, May 31, 2001, from 9:00 AM to 10:00 AM.'}

{'indice': 1921, 'pregunta': 'What is the total amount of the expense report for Brent A. Price that is ready for my approval?', 'respuesta': 'The total amount of the expense report is $462.06.'}

{'indice': 2037, 'pregunta': 'By what date must users of the ect.enron.com, ei.enron.com, and ees.enron.com email domains switch to the firstname.lastname@enron.com format?', 'respuesta': 'These domains will be decommissioned by the close of business on Friday, January 18th.'}

{'indice': 844, 'pregunta': 'Who

##Queries Largos

In [ ]:
correos_largos = df_beck_final[n_tokens > 200].sample(frac=1, random_state=42)

preguntas_largas = []
objetivo = 50
tamano_lote = 10
posicion = 0

while len(preguntas_largas) < objetivo and posicion < len(correos_largos):
    lote = correos_largos.iloc[posicion:posicion + tamano_lote]

    lote_correos = []
    for idx, fila in lote.iterrows():
        lote_correos.append({"indice": idx, "body": fila["body"]})

    resultado_lote = agente_generador_preguntas(lote_correos)
    preguntas_largas.extend(resultado_lote)

    posicion += tamano_lote
    print(f"Lote procesado (correos {posicion - tamano_lote}-{posicion}). Total acumulado: {len(preguntas_largas)}/{objetivo}")

    time.sleep(4)

print("Terminado")

Lote procesado (correos 0-10). Total acumulado: 8/50
Lote procesado (correos 10-20). Total acumulado: 16/50
Lote procesado (correos 20-30). Total acumulado: 23/50
Lote procesado (correos 30-40). Total acumulado: 30/50
Lote procesado (correos 40-50). Total acumulado: 40/50
Lote procesado (correos 50-60). Total acumulado: 50/50
Terminado


In [ ]:
for item in preguntas_largas[10:16]:
    print(item)
    print()

{'indice': 385, 'pregunta': 'What is my office phone number that I provided to the person in charge of the malfunctioning system?', 'respuesta': 'My office phone number is 35926.'}

{'indice': 995, 'pregunta': 'Why will I be out of the office on March 2nd and March 5th, 2001?', 'respuesta': "I will be out of the office on those dates to chaperone Meagan's drill team to a state competition in Dallas."}

{'indice': 1229, 'pregunta': 'What is my scheduled Outlook migration date according to the notification from the Outlook team?', 'respuesta': 'My scheduled Outlook migration date is the evening of April 23rd.'}

{'indice': 438, 'pregunta': 'What is the frequency and the location of the ISC Training and Communication Services Team staff meetings starting September 29, 2000?', 'respuesta': 'The meetings occur every second week on Friday in conference room #42c1.'}

{'indice': 28, 'pregunta': 'Which cities had ENA Energy Operations employees working on January 1st, 2000, to test systems?', 

##Preguntas con más de un Mensaje

In [ ]:
correos_multi = df_beck_final[df_beck_final["n_mensajes"] > 1].sample(frac=1, random_state=42)
print(f"Correos disponibles: {len(correos_multi)}")

preguntas_multi = []
objetivo = 30
tamano_lote = 10
posicion = 0

while len(preguntas_multi) < objetivo and posicion < len(correos_multi):
    lote = correos_multi.iloc[posicion:posicion + tamano_lote]

    lote_correos = []
    for idx, fila in lote.iterrows():
        lote_correos.append({"indice": idx, "body": fila["body"]})

    resultado_lote = agente_generador_preguntas(lote_correos)
    preguntas_multi.extend(resultado_lote)

    posicion += tamano_lote
    print(f"Lote procesado (correos {posicion - tamano_lote}-{posicion}). Total acumulado: {len(preguntas_multi)}/{objetivo}")

    time.sleep(4)

print("Terminado")

Correos disponibles: 261
Lote procesado (correos 0-10). Total acumulado: 5/30
Lote procesado (correos 10-20). Total acumulado: 14/30
Lote procesado (correos 20-30). Total acumulado: 22/30
Lote procesado (correos 30-40). Total acumulado: 26/30
Lote procesado (correos 40-50). Total acumulado: 36/30
Terminado


In [ ]:
preguntas_multi = preguntas_multi[:30]
print(len(preguntas_multi))

30


In [ ]:
for item in preguntas_multi[10:16]:
    print(item)
    print()

{'indice': 791, 'pregunta': "Who was the individual recognized as a superstar for her efforts in securing gifts for the Office of the Chairman Holiday Assistant's Party?", 'respuesta': 'Heather Choate.'}

{'indice': 74, 'pregunta': 'What specific time did I propose for a meeting on Friday, February 7th?', 'respuesta': 'I proposed meeting at 2:00 pm.'}

{'indice': 1876, 'pregunta': 'Did Fernley Dyson confirm that Mike is included on the list of key employees for London operations regarding retention?', 'respuesta': 'Yes, Fernley Dyson confirmed that they have provided a list of key employees and Mike is on that list.'}

{'indice': 1699, 'pregunta': 'What is the primary focus of the role mapping exercise between ENW and the EES Services Group that Dan Leff is leading?', 'respuesta': 'The primary focus is on the areas of the business managed by Beth and Evan.'}

{'indice': 516, 'pregunta': 'What is the scheduled date, time, and location for my presentation at the Global Operations Control

##Preguntas sin Respuesta

In [ ]:
preguntas_sin_respuesta = [
    "What is the capital of Australia?",
    "How many players are on a soccer team on the field at once?",
    "What ingredients are needed to make a classic margherita pizza?",
    "Who won the FIFA World Cup in 2026?",
    "What is the boiling point of water in Celsius?",
    "Which planet is closest to the Sun?",
    "What year did the Titanic sink?",
    "How do you make a traditional French omelette?",
    "What is the tallest mountain in the world?",
    "Which TV show features a character named Walter White?",
    "What is the recipe for a classic Caesar salad dressing?",
    "How many time zones does Russia span?",
    "What team won the Super Bowl in 2024?",
    "What is the chemical symbol for gold?",
    "Which country hosted the 2016 Summer Olympics?",
    "How long does it take to bake a standard loaf of sourdough bread?",
    "What is the currency used in Japan?",
    "Who directed the movie Inception?",
    "What is the average lifespan of a golden retriever?",
    "Which ocean is the largest by surface area?",
]

filas_sin_respuesta = []
for pregunta in preguntas_sin_respuesta:
    filas_sin_respuesta.append({
        "tamano_correo": "sin_respuesta",
        "id_correo": None,
        "subject": None,
        "correo_original": None,
        "n_mensajes": None,
        "pregunta": pregunta,
        "respuesta_esperada": "No information available in the dataset",
    })

##Unificar

In [ ]:
def construir_filas(preguntas, etiqueta):
    filas = []
    for item in preguntas:
        idx = item["indice"]
        fila_original = df_beck_final.loc[idx]
        filas.append({
            "tamano_correo": etiqueta,
            "id_correo": idx,
            "subject": fila_original["subject"],
            "correo_original": fila_original["body"],
            "n_mensajes": fila_original["n_mensajes"],
            "pregunta": item["pregunta"],
            "respuesta_esperada": item["respuesta"],
        })
    return filas

todas_las_filas = []
todas_las_filas.extend(construir_filas(preguntas_cortas, "corto"))
todas_las_filas.extend(construir_filas(preguntas_intermedias, "intermedio"))
todas_las_filas.extend(construir_filas(preguntas_largas, "largo"))
todas_las_filas.extend(construir_filas(preguntas_multi, "multi_mensaje"))
todas_las_filas.extend(filas_sin_respuesta)

df_preguntas = pd.DataFrame(todas_las_filas)
print(len(df_preguntas))

df_preguntas.to_csv("/content/drive/MyDrive/TFM/banco_preguntas_v2.csv", index=False)

200


In [ ]:
df_preguntas = pd.read_csv("/content/drive/MyDrive/TFM/banco_preguntas_v2.csv")
df_preguntas.sample(5)

,tamano_correo,id_correo,subject,correo_original,n_mensajes,pregunta,respuesta_esperada
95,intermedio,2002.0,Re: FW: information for bert stromquist,[2001-12-31 23:08:03+00:00] beth.apollo@enron....,1.0,Who is aware of the boxes ready to be transfer...,"Pat Weatherspoon is aware of the boxes, and Be..."
5,corto,903.0,Beth's location,[2001-01-16 12:49:00+00:00] sally.beck@enron.c...,1.0,When is Beth Apollo starting her new role in H...,"Beth Apollo is starting on January 29th, and I..."
74,intermedio,1674.0,RE: A QUICK NOTE,[2001-10-28 22:20:30+00:00] sally.beck@enron.c...,1.0,Which team will the recipient of my October 28...,The recipient will be joining Beth Apollo's ri...
45,corto,1039.0,Re: Lunch w/ Cindy Olson,[2001-03-09 11:40:00+00:00] sally.beck@enron.c...,1.0,Did I confirm my availability for the meeting ...,"Yes, I confirmed that I can make March 30th wo..."
122,largo,2042.0,REMINDER - EGM Organizational Post Petition M...,[2002-01-04 21:18:20+00:00] judy.zoch@enron.co...,1.0,When and where is the EGM Organizational Post ...,"The meeting is scheduled for Monday, January 7..."


In [ ]:
df_beck_final.iloc[903]

,903
subject,Beth's location
from_addr,sally.beck@enron.com
fecha_primer_mensaje,2001-01-16 12:49:00+00:00
fecha_ultimo_mensaje,2001-01-16 12:49:00+00:00
body,[2001-01-16 12:49:00+00:00] sally.beck@enron.c...
n_mensajes,1


In [ ]:
correos_largos_banco = df_preguntas[df_preguntas["tamano_correo"] == "largo"]["id_correo"]

superan_256 = 0
for idx in correos_largos_banco:
    if n_tokens.loc[idx] > 256:
        superan_256 += 1

print(f"De {len(correos_largos_banco)} preguntas 'largas', {superan_256} corresponden a correos que superan 256 tokens (necesitaron chunking).")

De 50 preguntas 'largas', 35 corresponden a correos que superan 256 tokens (necesitaron chunking).


In [ ]:
df_preguntas = pd.read_csv("/content/drive/MyDrive/TFM/banco_preguntas_v2.csv")

In [ ]:
j=0

In [ ]:
j=j+5
for i in range(j-5,j):
    idx = df_preguntas.index[i]
    print(idx, df_preguntas["id_correo"].iloc[i], df_preguntas["pregunta"].iloc[i])
    print()

60 1845.0 Who will be covering for Lisa Costello and Mark Pickering while Lisa is on maternity leave starting November 19, 2001?

61 1411.0 What is the rescheduled date and time for the EA Staff Meeting that was moved due to the Memorial Holiday?

62 1921.0 What is the total amount of the expense report for Brent A. Price that is ready for my approval?

63 2037.0 By what date must users of the ect.enron.com, ei.enron.com, and ees.enron.com email domains switch to the firstname.lastname@enron.com format?

64 844.0 Who will be making the presentation for Rick Causey at the Analyst Orientation on January 19 because I will be out of town?



In [ ]:
correos_no_usados2 = correos_cortos.iloc[160:180]

for idx, fila in correos_no_usados2.iterrows():
    print(f"--- {idx} ---")
    print(fila["body"])
    print()

--- 1526 ---
[2001-10-03 22:04:06+00:00] sally.beck@enron.com:
Please print for Sally.

Thanks,

--- 588 ---
[2000-11-14 15:52:00+00:00] sally.beck@enron.com:
Please put this on my calendar.  I can't get this to print and it is too 
small for me to read!

--- 1585 ---
[2001-10-15 20:53:04+00:00] greg.piper@enron.com:
Whatever happended to all this?  Does Pickering have RisktRAC?

GP

--- 655 ---
[2000-11-28 18:44:00+00:00] sally.beck@enron.com:
Flight times are good.  Please insure that my hotel reservation is for a 
non-smoking room.  Thanks.  --Sally

--- 1840 ---
[2001-11-15 23:08:19+00:00] sally.beck@enron.com:
Attached is list of questions.  How soon could we have answers to these?  I would appreciate any help that you could give me on this.  --Sally

--- 1420 ---
[2001-05-31 15:54:45+00:00] adarsh.vakharia@enron.com:
Please see attached.

--- 1201 ---
[2001-04-11 18:15:00+00:00] sally.beck@enron.com:
This will be an important meeting for me to attend.  Let me know when this 
gets

In [ ]:
reemplazos = {
    19: {
        "id_correo": 655,
        "pregunta": "What did I request regarding my hotel room reservation?",
        "respuesta": "I requested that my hotel reservation be for a non-smoking room.",
    },
    23: {
        "id_correo": 1965,
        "pregunta": "What names still need to be checked according to Marla Barnard's summary?",
        "respuesta": "A few names still need to be checked with Bob Hall and Jenny.",
    },
    45: {
        "id_correo": 1730,
        "pregunta": "Where and to whom did I ask to fax the letterhead document?",
        "respuesta": "I asked to fax it to Marilou at The Tremont House.",
    },
    47: {
        "id_correo": 540,
        "pregunta": "How many ASE's have I attended, and who will I ask to schedule a meeting?",
        "respuesta": "I have attended three ASE's, and I will ask Patti to schedule a meeting.",
    },
}

for pos, datos in reemplazos.items():
    fila_correo = df_beck_final.loc[datos["id_correo"]]
    df_preguntas.loc[pos, "id_correo"] = datos["id_correo"]
    df_preguntas.loc[pos, "subject"] = fila_correo["subject"]
    df_preguntas.loc[pos, "correo_original"] = fila_correo["body"]
    df_preguntas.loc[pos, "n_mensajes"] = fila_correo["n_mensajes"]
    df_preguntas.loc[pos, "pregunta"] = datos["pregunta"]
    df_preguntas.loc[pos, "respuesta_esperada"] = datos["respuesta"]

print(df_preguntas.loc[[19, 23, 45, 47]])

   tamano_correo  id_correo                                  subject  \
19         corto      655.0  Re: Beck/Sally 13Dec ENR-1081 Itinerary   
23         corto     1965.0                                  Summary   
45         corto     1730.0                                     ASAP   
47         corto      540.0                                  Re: ASE   

                                      correo_original  n_mensajes  \
19  [2000-11-28 18:44:00+00:00] sally.beck@enron.c...         1.0   
23  [2001-12-18 01:28:36+00:00] marla.barnard@enro...         1.0   
45  [2001-11-01 22:24:21+00:00] sally.beck@enron.c...         1.0   
47  [2000-11-06 13:16:00+00:00] sally.beck@enron.c...         1.0   

                                             pregunta  \
19  What did I request regarding my hotel room res...   
23  What names still need to be checked according ...   
45  Where and to whom did I ask to fax the letterh...   
47  How many ASE's have I attended, and who will I...   

       

In [ ]:
df_preguntas.to_csv("/content/drive/MyDrive/TFM/banco_preguntas_v2.csv", index=False)

#Agente Generador

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google import genai
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')
client_gemini = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.1-flash-lite"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def agente_generador_respuesta_k1(pregunta, resultado_chroma, umbral_distancia=0.6):
    documento = resultado_chroma["documents"][0][0]
    distancia = resultado_chroma["distances"][0][0]

    if distancia > umbral_distancia:
        return {
            "respuesta": "No information available in the provided emails.",
            "prompt_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "tiempo_ejecucion": 0.0,
            "energia_wh": 0.0,
            "gwp_mgco2eq": 0.0,
        }

    prompt = f"""Eres el asistente personal de correo electrónico de Sally Beck. Sally es quien te está haciendo la pregunta directamente, así que cuando el correo describa algo que ella dijo, decidió o planeó, respóndele en primera persona ("I", "my"), como si le confirmaras su propia información — nunca uses "you"/"your" para referirte a ella, ni te refieras a ella en tercera persona ("Sally").

Responde la pregunta ÚNICAMENTE usando la información del siguiente correo.

CORREO:
{documento}

PREGUNTA: {pregunta}

Responde de forma directa, en el idioma del correo.
"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={"temperature": 0.2}
    )

    tiempo_ejecucion = time.time() - inicio
    energia = respuesta.impacts.energy.value
    gwp = respuesta.impacts.gwp.value

    return {
        "respuesta": respuesta.text.strip(),
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
        "energia_wh": round(energia.mean * 1000, 4),
        "gwp_mgco2eq": round(gwp.mean * 1_000_000, 4),
    }

In [ ]:
def agente_generador_respuesta_multi(pregunta, resultado_chroma, umbral_distancia=0.6):
    documentos = resultado_chroma["documents"][0]
    distancias = resultado_chroma["distances"][0]

    documentos_relevantes = []
    for doc, dist in zip(documentos, distancias):
        if dist <= umbral_distancia:
            documentos_relevantes.append(doc)

    if not documentos_relevantes:
        return {
            "respuesta": "No information available in the provided emails.",
            "prompt_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "tiempo_ejecucion": 0.0,
            "energia_wh": 0.0,
            "gwp_mgco2eq": 0.0,
        }

    contexto = "\n\n---\n\n".join(documentos_relevantes)
    n_correos = len(documentos_relevantes)

    prompt = f"""Eres el asistente personal de correo electrónico de Sally Beck. Sally es quien te está haciendo la pregunta directamente, así que cuando el correo describa algo que ella dijo, decidió o planeó, respóndele en primera persona ("I", "my"), como si le confirmaras su propia información — nunca uses "you"/"your" para referirte a ella, ni te refieras a ella en tercera persona ("Sally").

Recibirás {n_correos} correos recuperados de una base de datos vectorial por su similitud semántica con la pregunta. Estos correos no forman necesariamente un mismo hilo ni están relacionados entre sí: trátalos como fuentes independientes.

Responde la pregunta ÚNICAMENTE usando la información de los correos proporcionados.

CORREOS:
{contexto}

PREGUNTA: {pregunta}

Responde de forma directa, en el idioma de los correos.
"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={"temperature": 0.2}
    )

    tiempo_ejecucion = time.time() - inicio
    energia = respuesta.impacts.energy.value
    gwp = respuesta.impacts.gwp.value

    return {
        "respuesta": respuesta.text.strip(),
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
        "energia_wh": round(energia.mean * 1000, 4),
        "gwp_mgco2eq": round(gwp.mean * 1_000_000, 4),
    }

In [ ]:
df_preguntas = pd.read_csv("/content/drive/MyDrive/TFM/banco_preguntas_v2.csv")

In [ ]:
# Instalamos ChromaDB, que será nuestra base de datos vectorial
!pip install chromadb -q

import chromadb

cliente_completos = chromadb.PersistentClient(path="/content/drive/MyDrive/TFM/chroma_beck_A")
collection_completos = cliente_completos.get_or_create_collection(
    name="beck_mails_completos",
    metadata={"hnsw:space": "cosine"}
)
print("Documentos en Configuración A:", collection_completos.count())

cliente_resumenes = chromadb.PersistentClient(path="/content/drive/MyDrive/TFM/chroma_beck_B")
collection_resumenes = cliente_resumenes.get_or_create_collection(
    name="beck_resumenes",
    metadata={"hnsw:space": "cosine"}
)
print("Documentos en Configuración B:", collection_resumenes.count())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [ ]:
!pip install "ecologits[google-genai]" -q

from ecologits import EcoLogits
EcoLogits.init(providers=["google_genai"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 1.75.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


##Pruebas

In [ ]:
for i in range(3):
  fila_c1 = df_preguntas[df_preguntas["tamano_correo"] == "corto"].iloc[i]
  resultado_c1 = collection_completos.query(query_texts=[fila_c1["pregunta"]], n_results=1)
  respuesta_c1 = agente_generador_respuesta_k1(fila_c1["pregunta"], resultado_c1)
  print("Pregunta:", fila_c1["pregunta"])
  print("Esperada:", fila_c1["respuesta_esperada"])
  print("Generada:", respuesta_c1["respuesta"])
  print()

Pregunta: Who is the contact person listed in the contact list forwarded to Mari on December 21, 2001?
Esperada: The contact person listed is Christina Valdez, Assistant to Sally Beck.
Generada: The provided email does not contain information regarding a contact list forwarded to Mari on December 21, 2001.

Pregunta: What is the monthly burn rate for the six Valtech contractors working for CL as reported by Tom Gros on November 12, 2001?
Esperada: The monthly burn rate is about $110,000.
Generada: The monthly burn rate for the six Valtech contractors working for CL is about $110k, as I noted in the information provided by Tom Gros.

Pregunta: Who should be coordinated to perform the task and who needs to provide updates regarding Fernley's access?
Esperada: Scott Tholen's team should be coordinated to perform the task, and I need to be kept updated on when Fernley will have access.
Generada: I should coordinate Scott Tholen's team to perform the task, and I need to be kept updated on w

In [ ]:
for i in range(3):
    fila_i = df_preguntas[df_preguntas["tamano_correo"] == "intermedio"].iloc[i]
    resultado_i = collection_completos.query(query_texts=[fila_i["pregunta"]], n_results=1)
    respuesta_i = agente_generador_respuesta_k1(fila_i["pregunta"], resultado_i)
    print("Pregunta:", fila_i["pregunta"])
    print("Esperada:", fila_i["respuesta_esperada"])
    print("Generada:", respuesta_i["respuesta"])
    print()

Pregunta: Where and at what time will the daily merger update meetings be held starting November 13, 2001?
Esperada: The daily merger update meetings will be held at 5:00 pm in conference room ECS 04075.
Generada: The provided email does not contain information regarding the daily merger update meetings starting on November 13, 2001.

Pregunta: For which employees did I complete internal feedback forms in the PEP system as of December 29, 2000?
Esperada: I completed feedback forms for Brent Price, Bob Hall, and Peggy Hedstrom.
Generada: I completed internal feedback forms in the PEP system for Brent Price, Bob Hall, and Peggy Hedstrom.

Pregunta: Who needs to sign the headcount addition forms for the new positions, and what should they understand before signing?
Esperada: Fred and Scott Neal need to sign the forms, and they must understand the additional costs associated with these headcount requests.
Generada: Fred and Scott Neal need to sign the forms, and they need to understand the

In [ ]:
for i in range(3):
    fila_l = df_preguntas[df_preguntas["tamano_correo"] == "largo"].iloc[i]
    resultado_l = collection_completos.query(query_texts=[fila_l["pregunta"]], n_results=1)
    respuesta_l = agente_generador_respuesta_k1(fila_l["pregunta"], resultado_l)
    print("Pregunta:", fila_l["pregunta"])
    print("Esperada:", fila_l["respuesta_esperada"])
    print("Generada:", respuesta_l["respuesta"])
    print()

Pregunta: What are the specific topics and people I planned to discuss or meet with during my upcoming trip to the London office in January 2001?
Esperada: I planned to review the action plan for London books, replicate an Operational Risk session, discuss the Doorstep process, technology dependencies with Mark Pickering, planning for the February 12 & 13 ASE, and the operational pricing model. I also intended to meet with Michael Brown, Beth Apollo, Ted Murphy, and Fernley.
Generada: For my upcoming trip to the London office, I have planned the following:

*   **Review the action plan** to move the close and officialization of London books to the same day.
*   **Replicate an operational risk information session** for London counterparts, after reviewing what we are doing in Houston and modifying the information for Europe.
*   **Discuss the doorstep process** with business controllers in the London office who will participate in doorstep reviews this year, to clear up questions and ge

In [ ]:
for i in range(3):
  fila_c1 = df_preguntas[df_preguntas["tamano_correo"] == "corto"].iloc[i]
  resultado_c1 = collection_completos.query(query_texts=[fila_c1["pregunta"]], n_results=3)
  respuesta_c1 = agente_generador_respuesta_multi(fila_c1["pregunta"], resultado_c1)
  print("Pregunta:", fila_c1["pregunta"])
  print("Esperada:", fila_c1["respuesta_esperada"])
  print("Generada:", respuesta_c1["respuesta"])
  print()

Pregunta: Who is the contact person listed in the contact list forwarded to Mari on December 21, 2001?
Esperada: The contact person listed is Christina Valdez, Assistant to Sally Beck.
Generada: The contact person listed in the contact list I forwarded to Mari on December 21, 2001, is Christina Valdez, my assistant.

Pregunta: What is the monthly burn rate for the six Valtech contractors working for CL as reported by Tom Gros on November 12, 2001?
Esperada: The monthly burn rate is about $110,000.
Generada: The monthly burn rate for the six Valtech contractors working for CL is approximately $110k.

Pregunta: Who should be coordinated to perform the task and who needs to provide updates regarding Fernley's access?
Esperada: Scott Tholen's team should be coordinated to perform the task, and I need to be kept updated on when Fernley will have access.
Generada: I need to coordinate Scott Tholen's team to perform the task, and they are the ones who need to keep me updated on when Fernley w

In [ ]:
for i in range(3):
    fila_i = df_preguntas[df_preguntas["tamano_correo"] == "intermedio"].iloc[i]
    resultado_i = collection_completos.query(query_texts=[fila_i["pregunta"]], n_results=3)
    respuesta_i = agente_generador_respuesta_multi(fila_i["pregunta"], resultado_i)
    print("Pregunta:", fila_i["pregunta"])
    print("Esperada:", fila_i["respuesta_esperada"])
    print("Generada:", respuesta_i["respuesta"])
    print()

Pregunta: Where and at what time will the daily merger update meetings be held starting November 13, 2001?
Esperada: The daily merger update meetings will be held at 5:00 pm in conference room ECS 04075.
Generada: I have no information in the provided emails regarding the location or time for the daily merger update meetings starting on November 13, 2001.

Pregunta: For which employees did I complete internal feedback forms in the PEP system as of December 29, 2000?
Esperada: I completed feedback forms for Brent Price, Bob Hall, and Peggy Hedstrom.
Generada: As of December 29, 2000, I completed internal feedback forms in the PEP system for Brent Price, Bob Hall, and Peggy Hedstrom.

Pregunta: Who needs to sign the headcount addition forms for the new positions, and what should they understand before signing?
Esperada: Fred and Scott Neal need to sign the forms, and they must understand the additional costs associated with these headcount requests.
Generada: Fred and Scott Neal need to 

In [ ]:
for i in range(3):
    fila_l = df_preguntas[df_preguntas["tamano_correo"] == "largo"].iloc[i]
    resultado_l = collection_completos.query(query_texts=[fila_l["pregunta"]], n_results=3)
    respuesta_l = agente_generador_respuesta_multi(fila_l["pregunta"], resultado_l)
    print("Pregunta:", fila_l["pregunta"])
    print("Esperada:", fila_l["respuesta_esperada"])
    print("Generada:", respuesta_l["respuesta"])
    print()

Pregunta: What are the specific topics and people I planned to discuss or meet with during my upcoming trip to the London office in January 2001?
Esperada: I planned to review the action plan for London books, replicate an Operational Risk session, discuss the Doorstep process, technology dependencies with Mark Pickering, planning for the February 12 & 13 ASE, and the operational pricing model. I also intended to meet with Michael Brown, Beth Apollo, Ted Murphy, and Fernley.
Generada: During my upcoming trip to the London office, I have planned the following agenda:

**Topics to discuss:**
*   **London Books:** Review my action plan to move the close and officialization of London books to the same day.
*   **Operational Risk:** Replicate an operational risk information session for London counterparts, modifying the information used in Houston for the European office.
*   **Doorstep Process:** Discuss this process and get input on how to continually improve it, including meeting with bu

#Experimentos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google import genai
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')
client_gemini = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.1-flash-lite"

Mounted at /content/drive


In [ ]:
df_beck_final = pd.read_csv("/content/drive/MyDrive/TFM/dataset_beck_final.csv")
df_preguntas = pd.read_csv("/content/drive/MyDrive/TFM/banco_preguntas.csv")

In [ ]:
!pip install chromadb -q

import chromadb

cliente_completos = chromadb.PersistentClient(path="/content/drive/MyDrive/TFM/chroma_beck_A")
collection_completos = cliente_completos.get_or_create_collection(
    name="beck_mails_completos",
    metadata={"hnsw:space": "cosine"}
)
print("Documentos en Configuración A:", collection_completos.count())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 836.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
!pip install "ecologits[google-genai]" -q
from ecologits import EcoLogits
EcoLogits.init(providers=["google_genai"])

In [ ]:
resultados_a_k1 = []

for i, fila in df_preguntas.iterrows():
    pregunta = fila["pregunta"]

    resultado_chroma = collection_completos.query(query_texts=[pregunta], n_results=1)
    respuesta = agente_generador_respuesta(pregunta, resultado_chroma)

    resultados_a_k1.append({
        "id_pregunta": i,
        "pregunta": pregunta,
        "respuesta_esperada": fila["respuesta_esperada"],
        "tamano_correo": fila["tamano_correo"],
        "respuesta_generada": respuesta["respuesta"],
        "prompt_tokens": respuesta["prompt_tokens"],
        "output_tokens": respuesta["output_tokens"],
        "tiempo_ejecucion": respuesta["tiempo_ejecucion"],
        "energia_wh": respuesta["energia_wh"],
        "gwp_mgco2eq": respuesta["gwp_mgco2eq"],
    })

    if i % 20 == 0:
        pd.DataFrame(resultados_a_k1).to_csv("/content/drive/MyDrive/TFM/experimento_A_k1.csv", index=False)
        print(f"Progreso: {i}/{len(df_preguntas)}")

    time.sleep(4)

pd.DataFrame(resultados_a_k1).to_csv("/content/drive/MyDrive/TFM/experimento_A_k1.csv", index=False)
print("Terminado")